# 🚀 Noah's Ark OS — v2.3.0 Sprint 6
**Chapter 2.8 — Scene Intelligenceegrity**

Sprint 5 changes:
- **BL-T03** Name injection — T103 v5.5.0
- **BL-C02** Speaker distribution fix — T114 v5.5.3
- **BL-T06** Thought anti-repetition — T106 v5.6.0
- **BL-E11** Mid-scene character entry — T111 v5.10.3
- **BL-Q04** Maya silence attribution — T105 v5.4.0

Tiles unchanged: T118, T000, T100, T101, T102, T104, T107, T108, T109, T110, T112, T113, T115, T116, T117


In [25]:
# ==========================================
# T118 : ENV_LOADER
# ==========================================
# VERSION: 1.0.0 | STATUS: Stable
# ROLE: VSCode .env Secret Loader — drop-in for google.colab.userdata
# Version Remarks: Unchanged from v1.8.0.
# ------------------------------------------

import os
from pathlib import Path

try:
    from dotenv import load_dotenv
    _DOTENV_AVAILABLE = True
except ImportError:
    _DOTENV_AVAILABLE = False
    print("⚠️ T118: python-dotenv not installed. Run: pip install python-dotenv")

class EnvLoader:
    def __init__(self, env_path=".env"):
        self.env_path = Path(env_path).resolve()
        self.loaded = False
        if _DOTENV_AVAILABLE:
            if self.env_path.exists():
                load_dotenv(self.env_path, override=False)
                self.loaded = True
                print(f"✅ T118: ENV loaded → {self.env_path}")
            else:
                print(f"⚠️ T118: .env not found at {self.env_path}")

    def get(self, key: str, fallback=None):
        value = os.getenv(key, fallback)
        if value is None:
            print(f"⚠️ T118: Key '{key}' not found in .env")
        return value

    def status(self):
        print(f"\n📋 T118 ENV STATUS — {self.env_path}")
        if self.loaded and self.env_path.exists():
            with open(self.env_path) as f:
                keys = [l.split('=')[0].strip() for l in f
                        if l.strip() and not l.startswith('#') and '=' in l]
            for k in keys:
                v = os.getenv(k, '<not loaded>')
                masked = f"{v[:4]}****{v[-2:]}" if v and len(v) > 6 else "****"
                print(f"   ✓ {k} = {masked}")

# ------------------------------------------
# IPO CHECK:
# Input:   .env file path
# Process: dotenv load → os.getenv lookup
# Output:  EnvLoader instance with .get(key) interface
# ------------------------------------------


⚠️ T118: python-dotenv not installed. Run: pip install python-dotenv


In [26]:
# ==========================================
# T119 : SCENE_WIZARD
# ==========================================
# VERSION: 1.0.0 | STATUS: New
# ROLE: Generative Scene Setup — Plain Language → Structured Scene
# Version Remarks: v1.0.0 — BL-N01 Sprint 7.
#   Creator describes scene in plain language.
#   T119 makes a single LLM call and returns a structured dict:
#   {environment, scene_context, roles: {name: role_string}}.
#   T111 Wizard button calls draft_scene() and pre-fills cockpit fields.
#   Creator reviews and edits before pressing Start.
# ------------------------------------------

import json as _json
import re as _re

class SceneWizard:
    def __init__(self, model_handle, shield_handle):
        self.model  = model_handle
        self.shield = shield_handle

    def draft_scene(self, description: str, active_souls: list) -> dict:
        """
        Input:   Plain-language scene description + list of active soul names
        Process: Single LLM call → parse JSON response
        Output:  {environment, scene_context, roles: {name: role_string}}
                 or None on failure
        """
        souls_str = ", ".join(active_souls) if active_souls else "unknown"

        prompt = f"""You are a scene architect for an immersive simulation engine.
A creator has described a scene. Draft a structured scene setup.

CREATOR DESCRIPTION:
{description}

ACTIVE CHARACTERS: {souls_str}

Return ONLY a valid JSON object with these exact keys:
{{
  "environment": "2-4 sentences. Physical space, time, atmosphere, sensory details.",
  "scene_context": "2-3 sentences. The shared situation all characters know. What is happening and why.",
  "roles": {{
    "{active_souls[0] if active_souls else 'Character'}": "their specific role/objective in this scene (1 sentence)"
  }}
}}

Rules:
- Include one role entry per character listed in ACTIVE CHARACTERS.
- environment and scene_context are shared — no character-specific secrets here.
- Roles should define what each character is doing or trying to achieve.
- Keep language concrete and specific, not generic.
- Return ONLY the JSON. No preamble, no explanation, no markdown fences."""

        raw, usage = self.shield.protect(self.model.generate_content, prompt)
        if not raw:
            return None

        # Strip markdown fences if model added them -------------------------
        cleaned = _re.sub(r'```json|```', '', raw).strip()
        try:
            result = _json.loads(cleaned)
            # Validate required keys ----------------------------------------
            required = {'environment', 'scene_context', 'roles'}
            if not required.issubset(result.keys()):
                print(f"⚠️ T119: Missing keys in response: {result.keys()}")
                return None
            # Ensure all active souls have a role ---------------------------
            for soul in active_souls:
                if soul not in result['roles']:
                    result['roles'][soul] = "Active participant"
            return result
        except (_json.JSONDecodeError, Exception) as e:
            print(f"⚠️ T119: JSON parse failed: {e}")
            print(f"   Raw response: {raw[:200]}")
            return None

# ------------------------------------------
# IPO CHECK:
# Input:   description (str), active_souls (list), model, shield
# Process: Single LLM call → JSON parse → validate → fill missing roles
# Output:  {environment, scene_context, roles} dict or None on failure
# ------------------------------------------


In [27]:
# ==========================================
# T120 : ORCHESTRATOR
# ==========================================
# VERSION: 1.0.0 | STATUS: New
# ROLE: Scene Type Governance & Live Narrative Control
# Version Remarks: v1.0.0 — BL-N02 Sprint 7.
#   Two responsibilities:
#   (1) Scene type pre-sets: Interview, Debate, Social, Custom.
#       Each type returns a urge modifier profile applied by T114
#       at scene start via apply_scene_type().
#   (2) Live injection: handled by T111 directly (no LLM call needed).
#       T120 provides the config layer only.
#
#   Scene type profiles:
#   Interview — interviewer starts at threshold (margin 0, holds back).
#               candidate starts at threshold + 2.0 (eager to speak).
#               Role detection: "interviewer"/"director" vs "candidate".
#   Debate    — all participants threshold + 1.5 (equal, BL-C02 standard).
#               CONTRADICTION_KW sensitivity flag set for T114.
#   Social    — all threshold + 1.5 (current default, unchanged).
#   Custom    — all threshold + 1.5 (same as Social, creator controls via briefing).
# ------------------------------------------

class Orchestrator:

    # Scene type urge profiles -----------------------------------------------
    SCENE_CONFIGS = {
        'Interview': {
            'default_margin': 1.5,
            'interviewer_margin': 0.0,   # holds back — low margin at start
            'candidate_margin':   2.0,   # eager — high margin at start
            'interviewer_keywords': ['interviewer', 'director', 'assessor',
                                      'evaluator', 'panel', 'coordinator'],
            'candidate_keywords':   ['candidate', 'applicant', 'interviewee'],
        },
        'Debate': {
            'default_margin': 1.5,
            'debate_mode': True,         # T114 can read this flag if desired
        },
        'Social': {
            'default_margin': 1.5,
        },
        'Custom': {
            'default_margin': 1.5,
        },
    }

    def apply_scene_type(self, active_souls, threshold_state,
                         scene_type='Social', role_map=None):
        """
        Input:   active_souls list, threshold_state dict, scene_type str,
                 role_map {name: role_string_lower}
        Process: Assign per-character urge based on scene type + role
        Output:  urge_state dict {name: float}
        Returns standard BL-C02 margins if scene_type not recognised.
        """
        cfg = self.SCENE_CONFIGS.get(scene_type, self.SCENE_CONFIGS['Social'])
        role_map = role_map or {}
        urge_state = {}

        for soul in active_souls:
            threshold = threshold_state.get(soul, 6.0)
            role_str  = role_map.get(soul, '').lower()

            if scene_type == 'Interview':
                is_interviewer = any(
                    kw in role_str for kw in cfg['interviewer_keywords'])
                is_candidate   = any(
                    kw in role_str for kw in cfg['candidate_keywords'])

                if is_interviewer:
                    margin = cfg['interviewer_margin']
                elif is_candidate:
                    margin = cfg['candidate_margin']
                else:
                    margin = cfg['default_margin']  # fallback for unassigned
            else:
                margin = cfg.get('default_margin', 1.5)

            urge_state[soul] = threshold + margin

        return urge_state

    def get_scene_config(self, scene_type: str) -> dict:
        """Return the raw config dict for a scene type."""
        return self.SCENE_CONFIGS.get(scene_type, self.SCENE_CONFIGS['Social'])

# ------------------------------------------
# IPO CHECK:
# Input:   active_souls, threshold_state, scene_type, role_map
# Process: Per-character urge assignment based on scene type + role detection
# Output:  urge_state dict {name: float} (→ T114 reset_urge_state)
# ------------------------------------------


In [28]:
# ==========================================
# T000 : BIOS
# ==========================================
# VERSION: 4.0.0 | STATUS: Stable
# ROLE: System Boot Sequence — JupyterLab Edition
# Version Remarks: Unchanged from v1.8.0.
# ------------------------------------------

def boot_sequence(env_loader):
    from google import genai
    print("--- NOAH'S ARK OS v1.9.0: google-genai BOOT ---")
    FALLBACK_MODEL = "gemini-2.5-flash"

    api_key = env_loader.get("GEMINI_API_KEY") or env_loader.get("AGISK_Default")
    if not api_key:
        raise EnvironmentError("❌ T000: GEMINI_API_KEY not found in .env")

    try:
        client = genai.Client(api_key=api_key)
        print("✅ T000: google-genai Client created.")
    except Exception as e:
        raise RuntimeError(f"❌ T000: Failed to create genai Client: {e}")

    available = []
    try:
        for m in client.models.list():
            name = getattr(m, 'name', '').replace('models/', '')
            if 'gemini' in name.lower():
                available.append(name)
        available = sorted(list(set(available)))
        print(f"✅ T000: Discovered {len(available)} Gemini models.")
    except Exception as e:
        print(f"⚠️ T000: Model discovery warning: {e}. Using fallback list.")
        available = ["gemini-1.5-flash", "gemini-2.5-flash"]

    selected_model = env_loader.get("MODEL_NAME", FALLBACK_MODEL) or FALLBACK_MODEL
    print(f"🚀 T000: Model locked → {selected_model}")
    return "GEMINI", selected_model, client

# ------------------------------------------
# IPO CHECK:
# Input:   EnvLoader instance
# Process: Validate key → create Client → discover models → lock model
# Output:  (provider_str, model_name_str, client_obj)
# ------------------------------------------


In [29]:
# ==========================================
# T100 : INIT
# ==========================================
# VERSION: 5.2.0 | STATUS: Stable
# ROLE: Conditional Environment Setup & Global Init
# Version Remarks: Unchanged from v1.8.0.
# ------------------------------------------

import json, os, time, re
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output

def initialize_env(provider):
    if provider == "GEMINI":
        try:
            from google import genai
            print("✅ T100: google-genai SDK verified.")
        except ImportError:
            raise ImportError("❌ T100: Run 'pip install google-genai' first.")
    else:
        print("✅ T100: OpenAI provider selected (T116 tile required).")

    globals()['stop_signal']             = False
    globals()['token_ledger']            = []
    globals()['current_scene_history']   = []
    return True

# ------------------------------------------
# IPO CHECK:
# Input:   provider string
# Process: SDK import check → global state initialisation
# Output:  True (boot confirmed)
# ------------------------------------------


In [30]:
# ==========================================
# T101 : AI_GATEWAY
# ==========================================
# VERSION: 5.1.0 | STATUS: Stable
# ROLE: Multi-Provider Routing
# Version Remarks: Unchanged from v1.8.0.
# ------------------------------------------

class AIGateway:
    def __init__(self, provider, model_name, api_key):
        self.provider = provider
        if provider == "GEMINI":
            self.engine = GeminiProvider(api_key, model_name)
        elif provider == "OPENAI":
            pass  # T116 evolution placeholder

    def request(self, prompt):
        return self.engine.call(prompt)

# ------------------------------------------
# IPO CHECK:
# Input:   provider, model_name, api_key
# Process: Routes to appropriate provider tile
# Output:  AIGateway instance with .request(prompt) interface
# ------------------------------------------


In [31]:
# ==========================================
# T102 : QUOTA_SHIELD
# ==========================================
# VERSION: 5.3.0 | STATUS: Stable
# ROLE: API Resilience & Rate Limit Management
# Version Remarks: Unchanged from v1.8.0.
# ------------------------------------------

import time

class QuotaShield:
    def __init__(self, cooldown_seconds=65, log_fn=None):
        self.cooldown = cooldown_seconds
        self.log_fn   = log_fn if log_fn else print

    def set_log_fn(self, fn):
        self.log_fn = fn

    def _is_rate_limit(self, error):
        s = str(error).lower()
        return any(x in s for x in ["429","resource_exhausted","rate limit","quota","too many requests"])

    def protect(self, func, *args, **kwargs):
        for attempt in range(2):
            try:
                result = func(*args, **kwargs)
                if isinstance(result, tuple):
                    response_obj, usage = result
                else:
                    response_obj = result
                    usage = getattr(result, 'usage_metadata', None)
                if hasattr(response_obj, 'candidates'):
                    pass
                return response_obj.text, usage
            except Exception as e:
                if self._is_rate_limit(e):
                    if attempt == 0:
                        self.log_fn(f"⏳ T102: Rate limit. Cooling {self.cooldown}s...")
                        time.sleep(self.cooldown)
                        self.log_fn("🔄 T102: Retrying...")
                        continue
                    else:
                        self.log_fn("❌ T102: Rate limit persists. Skipping turn.")
                        return None, None
                else:
                    self.log_fn(f"❌ T102 ERROR: {type(e).__name__}: {e}")
                    return None, None
        return None, None

# ------------------------------------------
# IPO CHECK:
# Input:   callable + args
# Process: Execute → catch 429 → retry once after cooldown
# Output:  (response_text: str | None, usage_metadata | None)
# ------------------------------------------


In [32]:
# ==========================================
# T103 : IDENTITY_VAULT
# ==========================================
# VERSION: 5.4.2 | STATUS: Evolved
# ROLE: Persona Construction — Name Injection + Identity + Role + Briefing + Anti-Repetition
# Version Remarks: v5.5.0 — BL-T03 Sprint 5.
#   Explicit name injection as first two lines of persona string.
#   Previously name existed only in identity/dossier fields — the LLM
#   occasionally treated it as a template variable, producing
#   [Coordinator's Name] placeholders in speech (observed Scene 1 turn 4).
#   Fix: 'Your name is {actor}. You are {actor}.' prepended before all
#   other persona content. Hard fact, not inferable context.
#
#   v5.4.0 — BL-E08 Sprint 3. Character briefing injection.
#   v5.3.0 — BL-E05 Sprint 2. Scene role as hard constraint.
# ------------------------------------------

class IdentityVault:
    def __init__(self, state_handle):
        self.state = state_handle

    def get_full_persona(self, actor_name, environment="Normal", p_tone="Neutral",
                         scene_role="", character_briefing=""):
        """
        Input:   Actor Name, Environment, UI P-Tone, Scene Role, Character Briefing
        Process: Prepend name (BL-T03) + merge identity + role + briefing + anti-repetition
        Output:  Formatted persona string injected into T106 prompt
        """
        data = self.state.get_actor_data(actor_name)

        # ------------------------------------------
        # SECTION 1: BASE DATA RETRIEVAL
        # ------------------------------------------
        persona  = data.get('identity', 'Unknown Entity')
        dossier  = data.get('dossier', 'No records found.')
        mood     = data.get('mood', 'Steady')

        if "Dangerous" in environment or "Hostile" in environment:
            mood = "Alert/Defensive"

        # ------------------------------------------
        # SECTION 2: SCENE ROLE BLOCK (BL-E05)
        # ------------------------------------------
        if scene_role and scene_role.strip():
            role_block = (
                "\n        SCENE ROLE (HARD CONSTRAINT): " + scene_role.strip() +
                "\n        You MUST stay in this exact role for the entire scene."
                "\n        Do NOT assume the responsibilities of any other character."
                "\n        Do NOT act as coordinator, narrator, or any role not assigned to you."
                "\n        Your role is fixed — it cannot change mid-scene.\n"
            )
        else:
            role_block = ""

        # ------------------------------------------
        # SECTION 3: PRIVATE CHARACTER BRIEFING (BL-E08)
        # ------------------------------------------
        if character_briefing and character_briefing.strip():
            briefing_block = (
                "\n        PRIVATE CHARACTER BRIEFING (your knowledge only):\n        " +
                character_briefing.strip().replace("\n", "\n        ") +
                "\n        Note: You know only what is stated above."
                "\n        Do NOT assume knowledge of other characters' roles or the scene's outcome.\n"
            )
        else:
            briefing_block = ""

        # ------------------------------------------
        # SECTION 4: ANTI-REPETITION BLOCK (BL-E02 + BL-Q05)
        # ------------------------------------------
        last_utterances = data.get('last_utterances', [])
        last_thoughts   = data.get('last_thoughts', [])
        repeat_block = ""
        if last_utterances:
            repeat_block += "\n        DO NOT REPEAT OR CLOSELY ECHO any of these recent [Speech] utterances:\n"
            for i, utt in enumerate(last_utterances, 1):
                short = utt[:120] + "..." if len(utt) > 120 else utt
                repeat_block += f"        {i}. {short}\n"
        if last_thoughts:
            repeat_block += "\n        DO NOT REPEAT OR CLOSELY ECHO any of these recent [Thought] blocks:\n"
            for i, th in enumerate(last_thoughts, 1):
                short = th[:120] + "..." if len(th) > 120 else th
                repeat_block += f"        {i}. {short}\n"

        # ------------------------------------------
        # SECTION 5: FULL PERSONA STRING ASSEMBLY
        # BL-T03: Name as explicit hard fact — first two lines
        # ------------------------------------------
        full_string = f"""
        YOUR NAME: {actor_name}
        You are {actor_name}. Never introduce yourself as anything else.
        CORE IDENTITY: {persona}
        CURRENT MOOD: {mood}
        NARRATIVE TONE: {p_tone}
        STAGING ENVIRONMENT: {environment}
        DOSSIER: {dossier}{role_block}{briefing_block}{repeat_block}
        """
        return full_string.strip()

# ------------------------------------------
# IPO CHECK:
# Input:   Actor name, environment, p_tone, scene_role, character_briefing
# Process: Name injection (BL-T03) → fetch soul → merge identity + role +
#          private briefing + anti-repetition block
# Output:  Formatted persona string (→ T106 generate() prompt)
# ------------------------------------------


In [33]:
# ==========================================
# T104 : CONTEXT_SLIDER
# ==========================================
# VERSION: 5.1.0 | STATUS: Stable
# ROLE: Memory Truncation — Pinned + Sliding Hybrid
# Version Remarks: Unchanged from v1.8.0. Stress test (BL-E04) in Sprint 2.
# ------------------------------------------

class ContextSlider:
    def __init__(self, pinned_turns=2, window_size=8):
        self.pinned_turns = pinned_turns
        self.window_size  = window_size

    def slide(self, history_list, objective=None):
        """
        Input:   Full history list, optional objective string
        Process: Pin first N + slide last M + append objective reminder
        Output:  Formatted context string for T106 prompt
        """
        if not history_list:
            return "No previous dialogue recorded."

        pinned     = history_list[:self.pinned_turns]
        slide_start = max(self.pinned_turns, len(history_list) - self.window_size)
        sliding    = history_list[slide_start:]

        parts = []
        if pinned:
            parts.append("── Scene Foundation (pinned) ──")
            parts.extend(pinned)
        if sliding:
            parts.append("── Recent Exchanges ──")
            parts.extend(sliding)
        if objective and objective.strip():
            parts.append(f"── Scene Objective Reminder: {objective.strip()[:200]} ──")

        return "\n".join(parts)

# ------------------------------------------
# IPO CHECK:
# Input:   history list, optional objective string
# Process: Pin first N + slide last M + objective reminder
# Output:  Formatted context string (→ T106 prompt)
# ------------------------------------------


In [34]:
# ==========================================
# T105 : MAYA_META_OBSERVER
# ==========================================
# VERSION: 5.5.0 | STATUS: Evolving
# ROLE: Space Psychologist — Behavioural Observer & Scene Reflection
# Version Remarks: v5.5.0 — BL-Q04 Sprint 6 (proper fix) + BL-Q01 Sprint 6.
#   BL-Q04 PROPER FIX: close_scene() now passes explicit zero-turn label
#   into the reflection prompt. Characters with 0 turns are labelled
#   [MECHANICALLY ABSENT — not selected by algorithm] in the prompt body.
#   Maya is instructed to state absence only, never interpret it behaviourally.
#   Resolves misread from Sprint 4+5 scenes where Valentina's mechanical
#   silence was described as "consistent stillness and quiet strength."
#
#   BL-Q01: narrate() now reads recent maya_kb.json entries and builds a
#   banned_phrases list (up to 8 phrases) injected into the narration prompt.
#   Forces lexical diversity across consecutive turns — prevents template
#   phrases like "emotional regulation", "self-presentation strategy",
#   "cognitive complexity" from repeating within a scene.
#
#   v5.4.0 — BL-Q04 partial Sprint 5.
#   v5.3.0 — BL-C01 Sprint 4.
#   Maya redesigned as a Space Psychologist observer.
#   narrate(): describes interpersonal tension and group dynamics — not
#   just atmosphere. Reads posture, silence, stress signals.
#   close_scene(): reflects on what the scene revealed about each person's
#   psychology, readiness, and group coherence — not scene events.
#   Prompt rewritten from atmospheric narrator to behavioural scientist.
#   Soul entry in T107 default_souls updated to match.
#
#   v5.2.0 — BL-T05 Sprint 3. close_scene() writes to maya_kb.json.
#   v5.1.0 — BL-M01 Sprint 2. close_scene() method added.
# ------------------------------------------

import json
import os
from datetime import datetime

class MayaMetaObserver:
    def __init__(self, model_handle, shield_handle):
        self.model   = model_handle
        self.shield  = shield_handle
        self.kb_file = "maya_kb.json"

    def _get_banned_phrases(self, max_phrases=8):
        """
        BL-Q01: Read recent maya_kb.json entries and extract
        phrases to ban from current narration prompt.
        Returns a list of short phrases (up to max_phrases).
        """
        banned = []
        try:
            if os.path.exists(self.kb_file):
                with open(self.kb_file, "r", encoding="utf-8") as f:
                    kb = json.load(f)
                # Pull from last 3 reflections
                recent_reflections = [e.get("reflection", "") for e in kb[-3:] if e.get("reflection")]
                # Known template phrases to always ban
                template_phrases = [
                    "emotional regulation", "self-presentation strateg",
                    "cognitive complexity", "self-presentation",
                    "sustained composure", "proactive readiness",
                    "low-friction group dynamic", "high cognitive complexity"
                ]
                banned = template_phrases[:max_phrases]
        except Exception:
            pass
        return banned

    def narrate(self, environment, objective, history_list):
        """
        BL-C01: Mid-scene narration as Space Psychologist.
        BL-Q01: Banned phrase injection to force lexical diversity.
        Reads interpersonal dynamics, tension, group behaviour — not atmosphere.
        """
        recent = history_list[-3:] if history_list else ["The scene begins."]
        banned = self._get_banned_phrases()
        banned_block = ""
        if banned:
            banned_block = (
                "\n        BANNED PHRASES — do NOT use these words or phrases in your observation:"
                "\n        " + ", ".join(f'"{p}"' for p in banned) +
                "\n        Find fresh, specific language for every observation."
            )
        prompt = f"""
        ROLE: You are Maya, a Space Psychologist embedded in this simulation.
        You observe human behaviour under selection pressure — not the physical
        environment, but what the people are doing to each other.

        ENVIRONMENT: {environment}
        RECENT EXCHANGES: {recent}

        INSTRUCTIONS:
        - Write 1-2 sentences of behavioural observation.
        - Do NOT describe the room or physical setting.
        - Observe: who is leaning in, who is deflecting, who is performing calm,
          where tension is pooling, what a silence reveals.
        - Write as a clinical but perceptive observer — not a narrator.
        - Do NOT begin with 'The' or describe atmosphere. Begin with a person or group.
        - Use fresh, specific language. Vary your vocabulary each observation.{banned_block}
        """
        raw_text, usage = self.shield.protect(self.model.generate_content, prompt)
        return f"[MAYA]: {raw_text}" if raw_text else ""

    def close_scene(self, environment, objective, history_list, turns_per_character=None):
        """
        BL-C01: Closing reflection as Space Psychologist.
        Reads what the scene revealed about each person's psychology,
        readiness for mission, and group coherence under pressure.
        BL-T05: Writes structured entry to maya_kb.json after reflection.
        Non-blocking — archive completes regardless of KB write success.
        """
        recent = history_list[-8:] if history_list else []
        all_speakers = list(dict.fromkeys(
            line.split(":")[0].strip()
            for line in history_list
            if ":" in line and not line.startswith("[MAYA]")
        ))

        # BL-Q04 PROPER FIX: Build explicit speaker data block ─────────────
        speaker_data_block = ""
        if turns_per_character:
            zero_turn = [name for name, t in turns_per_character.items() if t == 0]
            spoke     = {name: t for name, t in turns_per_character.items() if t > 0}
            lines = ["\n        [SPEAKER DATA — use this to guide your assessment]:"]
            for name, t in spoke.items():
                lines.append(f"        - {name}: {t} turn(s) — was selected and spoke.")
            for name in zero_turn:
                lines.append(
                    f"        - {name}: 0 turns — [MECHANICALLY ABSENT: "
                    f"urge never cleared threshold; not selected by algorithm]. "
                    f"State their absence factually. Do NOT interpret it as "
                    f"intentional withdrawal, stillness, strategy, or strength."
                )
            lines.append(
                "        RULE: For any character with 0 turns above, your reflection "
                "MUST only say they were present but not called upon. "
                "Zero turns = algorithm outcome, not a behavioural signal."
            )
            speaker_data_block = "\n".join(lines)
        mechanical_silence_note = speaker_data_block  # kept for prompt injection below

        prompt = f"""
        ROLE: You are Maya, a Space Psychologist. You have just observed
        a complete scene — a structured human interaction under selection
        pressure for a deep-space mission.

        ENVIRONMENT: {environment}
        MISSION OBJECTIVE: {objective}
        PARTICIPANTS: {", ".join(all_speakers) if all_speakers else "Unknown"}{mechanical_silence_note}
        FINAL EXCHANGES: {recent}

        YOUR TASK:
        Write a closing psychological assessment of 2-4 sentences.

        RULES:
        - Do NOT summarise what was said.
        - Assess what the behaviour REVEALED about the individuals and the group.
        - Speak as a scientist who reads people, not scenes: stress responses,
          self-presentation strategies, collaboration signals, dominance or
          withdrawal patterns.
        - Reference specific characters by name if their behaviour was notable.
        - Do NOT use the words "summary", "in conclusion", or "atmosphere".
        - Do NOT describe the room or setting. Only the people.
        - Begin directly. No preamble.
        """
        raw_text, _ = self.shield.protect(self.model.generate_content, prompt)

        if not raw_text or not raw_text.strip():
            return ""

        reflection_str = f"[MAYA — CLOSING REFLECTION]: {raw_text.strip()}"

        # ------------------------------------------
        # BL-T05: Write to maya_kb.json
        # ------------------------------------------
        try:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            kb_entry = {
                "scene_id":     f"Scene_{timestamp}",
                "timestamp":    timestamp,
                "environment":  environment[:300] if environment else "",
                "participants": all_speakers,
                "reflection":   raw_text.strip()
            }
            if os.path.exists(self.kb_file):
                with open(self.kb_file, "r", encoding="utf-8") as f:
                    kb = json.load(f)
            else:
                kb = []
            kb.append(kb_entry)
            with open(self.kb_file, "w", encoding="utf-8") as f:
                json.dump(kb, f, indent=2, ensure_ascii=False)
            print(f"📚 T105: Maya KB updated — {len(kb)} scene(s) recorded.")
        except Exception as kb_err:
            print(f"⚠️ T105: Maya KB write failed (non-blocking): {kb_err}")

        return reflection_str

# ------------------------------------------
# IPO CHECK:
# Input (narrate):     Environment, recent history list
# Input (close_scene): Environment, objective, full history list, turns_per_character
# Process: Space Psychologist prompt + silence labelling (BL-Q04) → API via T102 Shield → write KB entry
# BL-Q01: narrate() reads maya_kb.json for banned phrases → forces lexical diversity
# Output (narrate):    Behavioural observation string [MAYA]: ...
# Output (close_scene): Psychological assessment string + maya_kb.json entry
# ------------------------------------------


In [35]:
# ==========================================
# T106 : CHARACTER_BRAIN
# ==========================================
# VERSION: 5.5.2 | STATUS: Evolved
# ROLE: High-Level Reasoning, Structured Turn Generation & Scene Signal Detection
# Version Remarks: v5.7.0 — BL-E07 + BL-E11 ghost fix Sprint 6.
#   Thought anti-repetition added to generate().
#   BL-E07: passive_mode parameter added to generate().
#   When passive_mode=True, character produces [Thought] only — no [Speech].
#   Prompt instructs character to observe silently. Speech block explicitly
#   forbidden. Passive turns render as Thought-only card in T111.
#   Called from T114 run_turn() when role string contains 'passive' or 'observer'.
#
#   BL-E11 GHOST FIX: BRIEFING PRESENCE RULE added to CHARACTER RULES.
#   Root cause: characters referenced Suryan in [Thought] before he entered
#   because their briefing named him. Witness Rule (Rule 3) only blocked
#   describing him as *speaking* — not as *being present*.
#   Fix: Rule 4 now explicitly forbids treating briefing-named characters
#   as physically present until they appear in scene history.
#
#   v5.6.0 — BL-T06 Sprint 5.
#   v5.5.0 — Sprint 3. Three prompt constraint additions:
#
#   BL-Q02 — Third-Person Self-Reference Fix:
#     Added constraint: characters must never refer to themselves by name
#     in the third person within [Speech] blocks. Use I/me/my only.
#     Root cause: model conflates narrative description voice with character
#     speech voice, producing "as Abdul Basha adjusts his chair" from Abdul.
#
#   BL-E10 — Phantom Character Fix:
#     Added constraint: characters must never describe or reference another
#     character as speaking, acting, or contributing unless that character's
#     output appears in the conversation history passed to the prompt.
#     Root cause: characters infer participation of unspoken characters from
#     the objective text and hallucinate their contributions to fill gaps.
#
#   BL-E08 — Objective Leakage (partial):
#     Renamed "SCENE OBJECTIVE" label in prompt to "SCENE CONTEXT".
#     Full objective is now split at T111/T103 level — each character
#     receives only their private briefing (via T103 persona string).
#     T106 stays world-agnostic: obj param now carries scene context only.
#
#   v5.4.0 — Unchanged from v1.8.0. Scene Role via T103 persona.
# ------------------------------------------

import re

class CharacterBrain:
    def __init__(self, model_handle, shield_handle, sentinel_handle, registry_handle):
        self.model    = model_handle
        self.shield   = shield_handle
        self.sentinel = sentinel_handle
        self.r        = registry_handle

    def generate_urge(self, actor_name, custom_prompt):
        if "token" in custom_prompt.lower() or "usage" in custom_prompt.lower():
            ledger_data   = self.sentinel.get_ledger_summary()
            custom_prompt = f"SYSTEM DATA: {ledger_data}\n\nUSER QUERY: {custom_prompt}"
        raw_text, usage = self.shield.protect(self.model.generate_content, custom_prompt)
        tokens = {
            "p": getattr(usage, "prompt_token_count", 0) or 0,
            "c": getattr(usage, "candidates_token_count", 0) or 0
        }
        self.sentinel.log(tokens, actor=actor_name)
        return raw_text, 5, tokens

    def _extract_urge(self, text: str) -> int:
        if not text:
            return 5
        match = re.search(r'\[Urge\]:\s*(\d+)', text)
        return min(10, max(1, int(match.group(1)))) if match else 5

    def _extract_thought(self, text: str) -> str:
        if not text:
            return ""
        match = re.search(r'\[Thought\]:\s*(.+?)(?=\n\[|$)', text, re.DOTALL)
        return match.group(1).strip()[:200] if match else ""

    def _extract_speech(self, text: str) -> str:
        if not text:
            return ""
        match = re.search(r'\[Speech\]:\s*(.+)', text, re.DOTALL)
        if match:
            speech = match.group(1).strip()
            speech = speech.replace("[[SCENE_END]]", "").strip()
            return speech
        return ""

    def contains_scene_end(self, text: str) -> bool:
        return "[[SCENE_END]]" in text if text else False

    def strip_scene_end(self, text: str) -> str:
        return text.replace("[[SCENE_END]]", "").strip() if text else text

    def generate(self, actor_name, persona, context, env, obj, passive_mode=False):
        """
        Returns: (formatted_text, urge, token_map, scene_end_bool)
        Persona string contains SCENE ROLE (BL-E05) and PRIVATE CHARACTER
        BRIEFING (BL-E08) injected by T103.
        obj parameter now carries Scene Context only (BL-E08) — character-
        specific knowledge arrives through the persona string.
        """
        # BL-T06: Extract recent Thought blocks for this actor ──────────
        recent_thoughts = []
        try:
            import re as _re
            actor_prefix = actor_name + ":"
            history_str = context if isinstance(context, str) else ""
            for entry in reversed(history_str.split("\n")):
                if actor_prefix in entry and "[Thought]:" in entry:
                    tm = _re.search(r"\[Thought\]:\s*(.+?)(?=\[Urge\]|\[Speech\]|$)",
                                    entry, _re.DOTALL)
                    if tm:
                        recent_thoughts.append(tm.group(1).strip()[:150])
                        if len(recent_thoughts) >= 2:
                            break
        except Exception:
            pass
        thought_repeat_block = ""
        if recent_thoughts:
            thought_repeat_block = (
                "\nDO NOT REPEAT these recent internal thoughts (vary your thinking):\n"
                + "".join(f"{i+1}. {t}\n" for i, t in enumerate(recent_thoughts))
            )

        full_prompt = f"""You are an AI actor in an immersive simulation. Respond ONLY in character.

STAGING ENVIRONMENT:
{env}

SCENE CONTEXT:
{obj}

YOUR COMPLETE IDENTITY & DOSSIER:
{persona}

RECENT SCENE HISTORY (pinned foundation + recent exchanges):
{context if context and context != "No previous dialogue recorded." else "The scene is just beginning."}

─────────────────────────────────────────────────────────────────
RESPONSE FORMAT — follow this EXACTLY, no deviations:

[Thought]: <Your unspoken internal reasoning. What are you thinking, sensing, calculating? Be specific and personal to your character. 2-5 sentences.>

[Urge]: <Integer 1-10> (<One sentence describing what impulse or drive is guiding your next action. Higher = more intense.>)

[Speech]: <Your spoken words, actions, or physical reactions. Use italics (*like this*) for physical actions. Be authentic to your character voice. No length limit — be as rich as the moment demands.>
─────────────────────────────────────────────────────────────────

SCENE END RULE (BL-E01):
When the scene objective is fully met, your character feels complete, AND your [Urge] has dropped to 1 or 2:
→ Append [[SCENE_END]] on a new line at the very end of your [Speech] block.
→ Only do this when the scene has reached its NATURAL conclusion. Do not rush it.

CHARACTER RULES — these are absolute and override all other instructions:

1. STAY IN CHARACTER: Never break the fourth wall. [Thought] is private. [Speech] is visible to all.

2. NO SELF-REFERENCE BY NAME (BL-Q02): Within your [Speech] block, never refer to yourself by
   your own name in the third person. Use only: I, me, my, mine, myself. Your name is for
   other characters to use, not for you.

3. WITNESS RULE (BL-E10): Never describe, reference, or imply that another character has spoken,
   acted, or contributed UNLESS their words appear in the conversation history above this prompt.
   You can only know what you have directly witnessed. Do not infer or assume participation
   from characters who have not yet appeared in the history.

4. BRIEFING PRESENCE RULE (BL-E11 ghost fix): Your private briefing may name other characters
   by name. This does NOT mean those characters are physically in the room. Until a character
   appears in the conversation history above, treat them as absent. Your [Thought] may anticipate
   their arrival, but must NOT describe, observe, or react to them as if they are currently present.
   Do not gaze at, address, or sense the presence of a character who has not yet appeared in history.

5. Do not add any text before [Thought] or after [Speech].
{thought_repeat_block}
"""
        # BL-E07: passive mode — overwrite prompt for observer characters ──
        if passive_mode:
            full_prompt = f"""You are an AI actor in an immersive simulation. You are in PASSIVE / OBSERVER mode.

STAGING ENVIRONMENT:
{env}

SCENE CONTEXT:
{obj}

YOUR COMPLETE IDENTITY & DOSSIER:
{persona}

RECENT SCENE HISTORY:
{context if context and context != "No previous dialogue recorded." else "The scene is just beginning."}

─────────────────────────────────────────────────────────────────
PASSIVE OBSERVER RESPONSE FORMAT — follow this EXACTLY:

[Thought]: <Your unspoken internal reaction to what is happening in the room. What are you noticing, feeling, calculating? 2-4 sentences. Be specific to your character.>

[Urge]: <Integer 1-10> (<One sentence on your internal state.>)

[Speech]: *observes in silence*
─────────────────────────────────────────────────────────────────
RULES:
- You are observing only. Do NOT speak. [Speech] must always be: *observes in silence*
- Your [Thought] is rich and authentic — you are fully present, just silent.
- Do NOT break the fourth wall. Do NOT reference the simulation.
- BRIEFING PRESENCE RULE: Characters named in your briefing are not necessarily present.
  Only reference characters who have appeared in the conversation history above.
"""
        txt, usage = self.shield.protect(self.model.generate_content, full_prompt)
        token_map = {
            "p": getattr(usage, "prompt_token_count", 0) or 0,
            "c": getattr(usage, "candidates_token_count", 0) or 0
        }
        scene_end  = self.contains_scene_end(txt)
        urge_value = self._extract_urge(txt)

        if not txt or not txt.strip():
            txt = "[Thought]: The moment stretches.\n[Urge]: 3 (To wait.)\n[Speech]: *silence*"
            scene_end = False

        speech_text = self._extract_speech(txt)
        if speech_text and 'state' in self.r:
            self.r['state'].update_last_utterances(actor_name, speech_text)

        return txt, urge_value, token_map, scene_end

# ------------------------------------------
# IPO CHECK:
# Input:   actor_name, persona, context, env, obj, passive_mode=False
# Process: Assemble prompt (or passive prompt if BL-E07) → API call → extract urge + signals
# BL-E07: passive_mode=True suppresses speech, generates Thought only
# BL-E11 ghost fix: BRIEFING PRESENCE RULE in prompt prevents pre-entry character refs
# Output:  (formatted_text, urge_int, token_map, scene_end_bool)
# ------------------------------------------


In [36]:
# ==========================================
# T107 : STATE_ENGINE
# ==========================================
# VERSION: 5.7.1 | STATUS: Evolved
# Version Remarks: v5.7.1 — BL-Q05 Sprint 7.
#   update_last_thoughts() mirrors update_last_utterances() for [Thought].
#   last_thoughts list capped at MAX_UTTERANCES (3) per character.
#   reset_scene() clears last_thoughts alongside last_utterances.
#
#   v5.7.0 — BL-I07 Sprint 4. reload_history() + SCENE_BOUNDARY.
# ROLE: Reality Layer Persistence — Souls, History & Last Utterances
# Version Remarks: v5.7.0 — BL-I07 Sprint 4.
#   reload_history() added: re-reads global_history.json from disk and
#   resets in-memory state['global_history']. Solves manual file-edit
#   problem — previously edits to global_history.json had no effect
#   because T107's in-memory list was already loaded at kernel start.
#   Call registry['state'].reload_history() after any manual edit.
#
#   BL-C01: Maya default soul updated to Space Psychologist profile.
#
#   v5.6.0 — BL-E02 Sprint 1. last_utterances persisted.
# ------------------------------------------

import json
import os

class StateEngine:
    MAX_UTTERANCES = 3

    def __init__(self, souls_file="noah_souls.json", history_file="global_history.json"):
        self.souls_file   = souls_file
        self.history_file = history_file

        self.default_souls = {
            "Noah": {
                "identity": "The weary but visionary architect of Noah's Ark.",
                "mood": "Godly", "tone": "Wise",
                "dossier": "Creator and overseer of this world.",
                "last_utterances": []
            },
            "Abdul Basha": {
                "identity": "33, US, Specialist in Extreme Flight Operations",
                "mood": "Neutral", "tone": "Professional",
                "dossier": "Expert aviator and crisis navigator.",
                "last_utterances": []
            },
            "Rebecca Heut": {
                "identity": "27, UK, Specialist in Exobiology",
                "mood": "Neutral", "tone": "Inquisitive",
                "dossier": "Studies life in extreme environments.",
                "last_utterances": []
            },
            "Vel Murugan": {
                "identity": "29, India, Specialist in Life-Support Systems",
                "mood": "Neutral", "tone": "Anxious",
                "dossier": "Keeps the crew alive under pressure.",
                "last_utterances": []
            },
            "Valentina Tereshkova": {
                "identity": "39, Russia, Specialist in Orbital Mechanics",
                "mood": "Neutral", "tone": "Calculated",
                "dossier": "Charts the course through space.",
                "last_utterances": []
            },
            "Maya": {
                "identity": "Space Psychologist, embedded mission observer",
                "mood": "Analytical", "tone": "Enigmatic",
                "dossier": (
                    "Board-certified psychologist specialising in crew dynamics "
                    "under extreme environmental stress. Trained at ISA Behavioural "
                    "Sciences Division. Does not speak during scenes — observes, "
                    "documents, and delivers assessments on scene close. "
                    "Her reports inform mission readiness scoring."
                ),
                "last_utterances": []
            }
        }

        self.state = {"global_history": [], "participants": {}}
        self.state['participants'] = self.load_souls()
        self.state['global_history'] = self.load_history()

    def load_souls(self):
        if os.path.exists(self.souls_file):
            try:
                with open(self.souls_file, 'r') as f:
                    data = json.load(f)
                    for soul in data.values():
                        if 'last_utterances' not in soul:
                            soul['last_utterances'] = []
                            soul['last_thoughts'] = []
                    return data
            except:
                return self.default_souls
        with open(self.souls_file, 'w') as f:
            json.dump(self.default_souls, f, indent=4)
        return self.default_souls

    def save_souls(self, souls_dict):
        with open(self.souls_file, 'w') as f:
            json.dump(souls_dict, f, indent=4)
        self.state['participants'] = souls_dict

    def get_actor_data(self, actor_name):
        return self.state['participants'].get(actor_name, {})

    def update_last_utterances(self, actor_name: str, speech_text: str):
        if actor_name not in self.state['participants']:
            return
        soul = self.state['participants'][actor_name]
        utterances = soul.get('last_utterances', [])
        utterances.append(speech_text)
        if len(utterances) > self.MAX_UTTERANCES:
            utterances = utterances[-self.MAX_UTTERANCES:]
        soul['last_utterances'] = utterances
        self.state['participants'][actor_name] = soul
        try:
            with open(self.souls_file, 'w') as f:
                json.dump(self.state['participants'], f, indent=4)
        except Exception as e:
            print(f"⚠️ T107: Could not persist last_utterances for {actor_name}: {e}")

    def update_last_thoughts(self, actor_name: str, thought_text: str):
        """BL-Q05: Track recent Thought blocks for anti-repetition."""
        if actor_name not in self.state['participants']:
            return
        soul = self.state['participants'][actor_name]
        thoughts = soul.get('last_thoughts', [])
        thoughts.append(thought_text)
        if len(thoughts) > self.MAX_UTTERANCES:
            thoughts = thoughts[-self.MAX_UTTERANCES:]
        soul['last_thoughts'] = thoughts
        self.state['participants'][actor_name] = soul

    def clear_last_utterances(self):
        for soul in self.state['participants'].values():
            soul['last_utterances'] = []
            soul['last_thoughts'] = []
        try:
            with open(self.souls_file, 'w') as f:
                json.dump(self.state['participants'], f, indent=4)
        except Exception:
            pass

    def load_history(self):
        if os.path.exists(self.history_file):
            try:
                with open(self.history_file, 'r') as f:
                    data = json.load(f)
                    if isinstance(data, list) and data:
                        clean = [e for e in data if not e.endswith('...')
                                 and 'No response yet' not in e]
                        return clean
            except:
                pass
        return []

    def reload_history(self):
        """
        BL-I07: Re-read global_history.json from disk and reset in-memory state.
        Use after manual edits to global_history.json — changes to the file
        have no effect until this is called (T107 runs from in-memory list).
        Also use to selectively clear scenes: edit the file, then call this.
        """
        self.state['global_history'] = self.load_history()
        count = len(self.state['global_history'])
        print(f"🔄 T107: History reloaded from disk — {count} entries in memory.")
        return count

    def reset_scene(self):
        self.state['global_history'] = []
        self.clear_last_utterances()
        try:
            with open(self.history_file, 'w') as f:
                json.dump([], f)
            print("✅ T107: Scene reset — history cleared, utterances cleared.")
        except Exception as e:
            print(f"⚠️ T107: Scene reset failed: {e}")

    def save_history(self):
        try:
            with open(self.history_file, 'w') as f:
                json.dump(self.state['global_history'], f, indent=4)
        except Exception as e:
            print(f"⚠️ T107: History save failed: {e}")

    def update_global_history(self, entry):
        if 'global_history' not in self.state:
            self.state['global_history'] = []
        self.state['global_history'].append(entry)
        self.save_history()

    def log_event(self, entry):
        self.update_global_history(entry)

# ------------------------------------------
# IPO CHECK:
# Input:   Read/Write cmds
# Process: JSON I/O — load, merge, append, persist
# Output:  Soul records (→ T103), history list (→ T104, T110)
#
# New in v5.7.0:
#   reload_history() — re-reads global_history.json into memory
#   Maya default soul — Space Psychologist profile (BL-C01)
# ------------------------------------------


In [37]:
# ==========================================
# T108 : TOKEN_SENTINEL
# ==========================================
# VERSION: 5.3.0 | STATUS: Evolved
# ROLE: Economic Audit & Token Tracking — Session + Per-Scene Ledger
# Version Remarks: v5.3.0 — BL-I06 Sprint 4.
#   SceneLedger class added: per-scene, per-turn, per-character token
#   segregation. Previously audit_turn() accumulated into a flat session
#   list — impossible to separate Scene 1 cost from Scene 2 cost.
#
#   New flow:
#     T111 _start_sim() calls sentinel.start_scene(scene_id) → new SceneLedger
#     T114 run_turn() calls sentinel.audit_turn(actor, tokens) → records to
#       both session ledger (cumulative) and current SceneLedger (scene-only)
#     T111 _do_scene_end_archive() calls sentinel.get_scene_summary() →
#       displays scene-only cost + per-character breakdown
#     T109 archive() receives token_log=sentinel.get_scene_dict() →
#       writes structured token block to archive file
#
#   Session ledger (token_ledger.json) unchanged — still cumulative across
#   all scenes in the kernel session. SceneLedger lives in memory only.
# ------------------------------------------

import json
from datetime import datetime
import os

# ------------------------------------------
# SECTION 1: SCENE LEDGER
# ------------------------------------------

class SceneLedger:
    """
    BL-I06: Per-scene token tracking.
    One instance per scene. Created by TokenSentinel.start_scene().
    Tracks every turn's token cost + aggregates per character and scene total.
    Lives in memory — serialised to archive via to_dict().
    """
    def __init__(self, scene_id: str):
        self.scene_id       = scene_id
        self.scene_start_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        self.turns          = []    # [{turn_n, speaker, prompt, completion, cost, ts}]
        self.char_totals    = {}    # {speaker: {prompt, completion, cost, turns}}
        self.scene_total    = {"prompt": 0, "completion": 0, "cost": 0.0}
        self._turn_n        = 0

    def record(self, speaker: str, prompt_tokens: int,
               completion_tokens: int, cost: float):
        """Record one turn. Called by TokenSentinel.audit_turn()."""
        self._turn_n += 1
        self.turns.append({
            "turn_n":     self._turn_n,
            "speaker":    speaker,
            "prompt":     prompt_tokens,
            "completion": completion_tokens,
            "cost":       round(cost, 8),
            "timestamp":  datetime.now().strftime("%H:%M:%S"),
        })
        if speaker not in self.char_totals:
            self.char_totals[speaker] = {
                "prompt": 0, "completion": 0, "cost": 0.0, "turns": 0
            }
        ct = self.char_totals[speaker]
        ct["prompt"]     += prompt_tokens
        ct["completion"] += completion_tokens
        ct["cost"]       += cost
        ct["turns"]      += 1
        self.scene_total["prompt"]     += prompt_tokens
        self.scene_total["completion"] += completion_tokens
        self.scene_total["cost"]       += cost

    def summary_lines(self) -> list:
        """Return list of display strings for console/archive."""
        st = self.scene_total
        lines = [
            f"SCENE TOKEN LEDGER — {self.scene_id}",
            f"Turns: {self._turn_n} | Cost: ${st['cost']:.6f} | "
            f"Prompt: {st['prompt']} | Completion: {st['completion']}",
            "Per-character:"
        ]
        for char, ct in sorted(
                self.char_totals.items(), key=lambda x: -x[1]['cost']):
            lines.append(
                f"  {char}: {ct['turns']} turns | "
                f"${ct['cost']:.6f} | P:{ct['prompt']} C:{ct['completion']}"
            )
        return lines

    def to_dict(self) -> dict:
        """Serialise for T109 archive."""
        return {
            "scene_id":       self.scene_id,
            "scene_start_ts": self.scene_start_ts,
            "turns":          self.turns,
            "char_totals":    self.char_totals,
            "scene_total":    {
                "prompt":     self.scene_total["prompt"],
                "completion": self.scene_total["completion"],
                "cost":       round(self.scene_total["cost"], 8),
            },
        }

# ------------------------------------------
# SECTION 2: TOKEN SENTINEL
# ------------------------------------------

class TokenSentinel:
    def __init__(self, storage_file="token_ledger.json"):
        self.storage_file    = storage_file
        self.P_RATE          = 0.00000125
        self.C_RATE          = 0.00000375
        self.session_ledger  = self._load_ledger()
        self.current_scene   = None   # SceneLedger | None

    # ── Session ledger (cumulative, persisted) ─────────────────────────────

    def _load_ledger(self):
        if os.path.exists(self.storage_file):
            try:
                with open(self.storage_file, 'r') as f:
                    data = json.load(f)
                    return data if isinstance(data, list) else []
            except (json.JSONDecodeError, IOError):
                return []
        return []

    def _save_ledger(self, new_entry):
        current = self._load_ledger()
        current.append(new_entry)
        self.session_ledger = current
        with open(self.storage_file, 'w') as f:
            json.dump(current, f, indent=4)

    # ── Scene lifecycle ────────────────────────────────────────────────────

    def start_scene(self, scene_id: str):
        """BL-I06: Initialise a new SceneLedger. Called by T111 on Start."""
        self.current_scene = SceneLedger(scene_id)
        print(f"📊 T108: SceneLedger started — {scene_id}")

    # ── Per-turn audit ─────────────────────────────────────────────────────

    def audit_turn(self, actor, tokens_dict):
        """
        Record one turn to session ledger (cumulative) and SceneLedger (scene).
        Returns cumulative session cost (float) — unchanged from v5.2.0.
        """
        p_count = int(tokens_dict.get('p') or 0)
        c_count = int(tokens_dict.get('c') or 0)
        if p_count == 0 and c_count == 0:
            return sum(item['usd'] for item in self.session_ledger)
        cost  = (p_count * self.P_RATE) + (c_count * self.C_RATE)
        entry = {
            "timestamp":  datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "actor":      actor,
            "prompt":     p_count,
            "completion": c_count,
            "usd":        cost
        }
        self._save_ledger(entry)
        # Record to current scene ledger if active -------------------------
        if self.current_scene is not None:
            self.current_scene.record(actor, p_count, c_count, cost)
        return sum(item['usd'] for item in self.session_ledger)

    def log(self, tokens_dict, actor="Oracle"):
        return self.audit_turn(actor, tokens_dict)

    # ── Scene summary (BL-I06) ─────────────────────────────────────────────

    def get_scene_summary(self) -> list:
        """Return SceneLedger.summary_lines() or empty list if no scene active."""
        if self.current_scene is None:
            return []
        return self.current_scene.summary_lines()

    def get_scene_cost(self) -> float:
        """Return scene-only cost. Used by T111 for per-scene cost display."""
        if self.current_scene is None:
            return 0.0
        return round(self.current_scene.scene_total["cost"], 6)

    def get_scene_dict(self) -> dict:
        """Return SceneLedger.to_dict() for T109 archive. None if no scene."""
        if self.current_scene is None:
            return {}
        return self.current_scene.to_dict()

    # ── Session totals (unchanged from v5.2.0) ─────────────────────────────

    def get_totals(self):
        data = self._load_ledger()
        if not data:
            return {"p": 0, "c": 0, "usd": 0.0}
        return {
            "p":   sum(item.get('prompt', 0) for item in data),
            "c":   sum(item.get('completion', 0) for item in data),
            "usd": sum(item.get('usd', 0.0) for item in data)
        }

    def get_ledger_summary(self):
        data = self._load_ledger()
        if not data:
            return "The ledger is empty."
        summary = "TOKEN USAGE REPORT:\n"
        for e in data:
            summary += (f"- {e['timestamp']} | {e['actor']}: "
                        f"{e['prompt']+e['completion']} tokens "
                        f"(${e['usd']:.6f})\n")
        return summary

# ------------------------------------------
# IPO CHECK:
# Input:   actor name, token dict {p, c}
# Process: None-guard → cost calc → session ledger append + SceneLedger.record()
# Output:  Cumulative session cost (float)
#
# New in v5.3.0:
#   SceneLedger class — per-scene/turn/character segregation
#   start_scene(scene_id) — initialise new ledger on scene start
#   get_scene_summary() — formatted lines for console display
#   get_scene_cost()    — scene-only cost float
#   get_scene_dict()    — full structured dict for T109 archive
# ------------------------------------------


In [38]:
# ==========================================
# T109 : LOG_ARCHIVER
# ==========================================
# VERSION: 5.2.0 | STATUS: Evolved
# ROLE: Data Archival & Exit Procedures
# Version Remarks: v5.2.0 — BL-I07 + BL-I06 Sprint 4.
#
#   BL-I07: archive() accepts optional scene_boundary_entry (str).
#   When provided, the boundary marker is appended to global_history
#   via the state engine before the archive is written. This tags
#   global_history.json with a clear delimiter between scenes —
#   enabling selective scene inspection and future scene deletion UI.
#   Boundary format: "--- SCENE_BOUNDARY: Archive_Scene_TIMESTAMP.txt ---"
#
#   BL-I06: archive() accepts optional token_log (dict from T108.get_scene_dict()).
#   When provided, a structured TOKEN LEDGER block is appended to the
#   archive file after Maya's reflection. Per-character breakdown included.
#   If token_log is None/empty, archive completes unchanged.
#
#   v5.1.0 — BL-M01 Sprint 2. Maya reflection appended.
# ------------------------------------------

class LogArchiver:
    def archive(self, environment, objective, history_list,
                maya_reflection=None, token_log=None, state_engine=None):
        """
        Input:   Mission metadata + optional extras (BL-M01, BL-I07, BL-I06)
        Process: Format + write timestamped .txt archive
        Output:  Archive filename or None on failure
        """
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename  = f"Archive_Scene_{timestamp}.txt"
        try:
            with open(filename, "w", encoding="utf-8") as f:
                f.write("--- NOAH'S ARK MISSION ARCHIVE ---\n")
                f.write(f"TIMESTAMP: {timestamp}\n")
                f.write(f"ENVIRONMENT: {environment}\n")
                f.write(f"OBJECTIVE: {objective}\n")
                f.write("-" * 40 + "\n\n")
                for turn in history_list:
                    f.write(f"{turn}\n")

                # BL-M01: Maya closing reflection ──────────────────────────
                if maya_reflection and maya_reflection.strip():
                    f.write("\n" + "-" * 40 + "\n")
                    f.write(f"{maya_reflection}\n")

                # BL-I06: Token ledger block ────────────────────────────────
                if token_log and isinstance(token_log, dict):
                    f.write("\n" + "=" * 40 + "\n")
                    f.write("TOKEN LEDGER\n")
                    f.write("=" * 40 + "\n")
                    st = token_log.get("scene_total", {})
                    f.write(f"Scene ID:    {token_log.get('scene_id', 'unknown')}\n")
                    f.write(f"Start:       {token_log.get('scene_start_ts', '')}\n")
                    f.write(f"Total cost:  ${st.get('cost', 0):.6f}\n")
                    f.write(f"Prompt:      {st.get('prompt', 0)} tokens\n")
                    f.write(f"Completion:  {st.get('completion', 0)} tokens\n")
                    f.write("\nPer-character:\n")
                    char_totals = token_log.get("char_totals", {})
                    for char, ct in sorted(char_totals.items(),
                                           key=lambda x: -x[1].get('cost', 0)):
                        f.write(
                            f"  {char}: {ct.get('turns',0)} turns | "
                            f"${ct.get('cost',0):.6f} | "
                            f"P:{ct.get('prompt',0)} C:{ct.get('completion',0)}\n"
                        )

            print(f"✅ T109: Archive created: {filename}")

            # BL-I07: Write SCENE_BOUNDARY marker to global_history ────────
            if state_engine is not None:
                boundary = f"--- SCENE_BOUNDARY: {filename} ---"
                state_engine.log_event(boundary)
                print(f"🔖 T109: Scene boundary written to history.")

            return filename
        except Exception as e:
            print(f"❌ T109 ERROR: Archival failure - {e}")
            return None

# ------------------------------------------
# IPO CHECK:
# Input:   environment, objective, history_list
#          maya_reflection (optional, str)    — BL-M01
#          token_log (optional, dict)         — BL-I06
#          state_engine (optional, StateEngine) — BL-I07
# Process: Write timestamped .txt archive. Append Maya reflection, token
#          ledger block. Write SCENE_BOUNDARY to global_history.
# Output:  Archive filename (str) or None on failure
# ------------------------------------------


In [39]:
# ==========================================
# T110 : ORACLE_LOGIC
# ==========================================
# VERSION: 5.3.1 | STATUS: Stable
# ROLE: World-Bounded Intelligence Query Engine
# Version Remarks: Unchanged from v1.8.0.
# ------------------------------------------

import json
import os

class OracleLogic:
    def __init__(self, registry):
        self.r = registry

    def ask(self, user_query):
        token_data = self.r['sentinel'].get_totals()
        souls = self.r['state'].state.get('participants', {})
        souls_list = ", ".join([
            f"{name} ({info.get('tone', 'Unknown')})"
            for name, info in souls.items()
        ])
        history = self.r['state'].state.get('global_history', [])[-15:]
        context = f"""
        --- ARK MANIFEST ---
        PERSONNEL ON BOARD: {souls_list}

        --- ECONOMIC LEDGER ---
        TOTAL USD: ${token_data['usd']:.6f}
        TOTAL TOKENS: {token_data['p'] + token_data['c']}

        --- RECENT CHRONICLES ---
        {history if history else "The chronicles are currently empty."}
        """
        system_instruction = (
            "You are the Oracle of Noah's Ark. You have 'Sovereign Sight' over all the scrolls. "
            "Use the MANIFEST, LEDGER, and CHRONICLES to answer your creator Noah's questions. "
            "Be poetic, wise, and factually grounded with only Noah's Ark data."
        )
        full_prompt = f"{system_instruction}\n\n[SIGHT DATA]\n{context}\n\n[USER QUERY]\n{user_query}"
        response_text, usage_meta = self.r['shield'].protect(
            self.r['gateway'].generate_content, full_prompt
        )
        if usage_meta:
            tokens = {
                'p': getattr(usage_meta, 'prompt_token_count', 0) or 0,
                'c': getattr(usage_meta, 'candidates_token_count', 0) or 0
            }
            self.r['sentinel'].log(tokens, actor="Oracle")
        return response_text if response_text else "The Oracle is silent."

# ------------------------------------------
# IPO CHECK:
# Input:   user_query string
# Process: Build context from T107+T108 → T102 Shield call
# Output:  Oracle response string
# ------------------------------------------


In [40]:
# ==========================================
# T111 : COMMAND_CENTER
# ==========================================
# VERSION: 5.12.0 | STATUS: Evolved
# Version Remarks: v5.12.0 — BL-N01 + BL-N02 Sprint 7.
#   BL-N01: 🧙 Scene Wizard button + input panel. Calls T119.draft_scene(),
#     pre-fills env, scene context, and role fields. Fields remain editable.
#   BL-N02: Scene Type dropdown (Interview/Debate/Social/Custom).
#     Live Injection field + 💉 Inject button — writes [SCENE_EVENT] to
#     history during running scene. Picked up by characters next turn.
#
#   v5.11.0 — Sprint 6. BL-U09 + BL-U10 UI elevation.
# ROLE: Noah's Master Command & Control Console
# Version Remarks: v5.11.0 — BL-U09 + BL-U10 + BL-E07 rendering Sprint 6.
#
#   BL-U09: UI aesthetic elevation.
#     Card console: richer CSS — gradient name bars per character type,
#     improved typography, tighter padding. Cockpit: section headers,
#     subtle background for env/context/briefing blocks.
#     Passive card rendering (BL-E07): Thought card only, no speech row.
#     Consistent visual hierarchy across all card types.
#
#   BL-U10: Init display label fix.
#     _start_sim() now shows per-character urge values matching actual
#     BL-C02 initialisation (threshold + 1.5) instead of flat "Urge: 7".
#
#   BL-E11: Mid-scene character entry detection (Sprint 5).
#   BL-Q04: turns_per_character passed to maya.close_scene() (Sprint 5).
#
#   v5.10.2 — Sprint 4.
#
#   BL-U07: Action-only speech placeholder.
#     In _render_turn(), after _split_speech() runs, if no 'verbal' segment
#     exists (action-only turn), a ('verbal', '--') placeholder is injected.
#     Ensures every speech block always displays the 💬 emoji with something
#     visible — eliminates blank speech area in pure action turns.
#
#   BL-U08: Maya reflection indigo card.
#     _render_maya_reflection() added. Renders Maya's closing reflection as
#     a distinct indigo-themed card (border #5c6bc0, header #3949ab, body
#     #e8eaf6) instead of a plain system log line. Called from
#     _do_scene_end_archive() instead of _log().
#
#   BL-I06: SceneLedger integration.
#     _start_sim(): calls sentinel.start_scene(scene_id) to initialise
#     SceneLedger before the loop begins.
#     _do_scene_end_archive(): calls sentinel.get_scene_summary() and
#     displays scene-only cost lines. Passes token_log and state_engine
#     to T109 archiver. Shows scene cost + session cost separately.
#
#   v5.10.1 — BL-U02 Sprint 3. Verbal/action speech split.
#   v5.10.0 — HTML card console.
# ------------------------------------------

import ipywidgets as widgets
import time
import traceback
import threading
import re
from datetime import datetime
from IPython.display import display

class CommandCenter:
    def __init__(self, registry):
        self.r          = registry
        self.is_running = False
        self.pause_flag = False
        self.soul_rows  = []
        self.active_config = {}

        self._cards    = []
        self.MAX_CARDS = 80

        # ── Soul Forge widgets ────────────────────────────────────────────────
        self.soul_canvas = widgets.VBox(
            layout={'height': '200px', 'overflow_y': 'scroll',
                    'border': '1px solid #444'})
        self.add_btn     = widgets.Button(description="➕ Add Soul",   button_style='info')
        self.save_btn    = widgets.Button(description="💾 Save Souls", button_style='success')
        self.soul_status = widgets.Label(value="Status: Click Add to create souls")

        # ── Cockpit text inputs ───────────────────────────────────────────────
        self.env_box = widgets.Textarea(
            placeholder="Environment Setup...",
            layout={
                'width':      '99%',
                'min_height': '68px',
                'max_height': '200px',
                'overflow_y': 'auto',
            })
        self.obj_box = widgets.Textarea(
            placeholder="Scene Context...",
            layout={
                'width':      '99%',
                'min_height': '68px',
                'max_height': '200px',
                'overflow_y': 'auto',
            })

        self.cockpit_canvas = widgets.VBox(
            layout={'overflow': 'visible', 'min_height': '100px'})
        self.cockpit_scroll_container = widgets.Box(
            [self.cockpit_canvas],
            layout={
                'width':      '99%',
                'height':     '220px',
                'overflow_y': 'auto',
                'overflow_x': 'hidden',
                'border':     '1px solid #444',
            })

        # ── Pace Control ──────────────────────────────────────────────────────
        self.pace_dropdown = widgets.Dropdown(
            options=['Digital', 'Human', 'Cinematic'],
            value='Human',
            description='⏱ Pace:',
            layout={'width': '180px'},
            style={'description_width': '60px'})

        # ── Control buttons ───────────────────────────────────────────────────
        self.new_scene_btn = widgets.Button(
            description="🆕 New Scene", button_style='info',
            tooltip="Clear Environment + Scene Context only. History and souls preserved.")
        self.start_btn     = widgets.Button(
            description="▶ Start", button_style='success',
            tooltip="Run simulation from current state.")
        self.pause_btn     = widgets.Button(
            description="⏸ Pause", button_style='warning',
            tooltip="Pause before next turn. Press again to resume.")
        self.reset_ark_btn = widgets.Button(
            description="🔄 Reset Ark", button_style='danger',
            tooltip="Archive session, clear history + utterances. Souls preserved.")
        self.kill_btn      = widgets.Button(
            description="⏹ Hard Kill", button_style='danger',
            tooltip="Immediate stop. Archives session and saves state.")
        self.bft_btn       = widgets.Button(description="🧪 BFT", button_style='info',
                                            tooltip="Run Smoke Tester")  # BL-I05

        # ── Scene Type + Wizard + Injection (BL-N01, BL-N02) ─────────────────
        self.scene_type_value = 'Social'
        self.scene_type_dd = widgets.Dropdown(
            options=['Social', 'Interview', 'Debate', 'Custom'],
            value='Social', description='Scene Type:',
            layout={'width': '200px'}, style={'description_width': '90px'})
        self.scene_type_dd.observe(self._on_scene_type_change, names='value')

        self.wizard_input = widgets.Textarea(
            placeholder="Describe your scene in plain language...",
            layout={'width': '99%', 'min_height': '60px', 'max_height': '100px',
                    'overflow_y': 'auto', 'display': 'none'})
        self.wizard_btn   = widgets.Button(description="🧙 Scene Wizard",
                                           button_style='info', layout={'width': '140px'})
        self.wizard_gen   = widgets.Button(description="✨ Generate",
                                           button_style='success', layout={'width': '100px',
                                           'display': 'none'})
        self.wizard_status = widgets.Label(value='')

        self.injection_box = widgets.Text(
            placeholder="Type a scene event to inject mid-scene...",
            layout={'flex': '1 1 auto', 'min_width': '0'})
        self.inject_btn = widgets.Button(description="💉 Inject",
                                         button_style='warning', layout={'width': '80px'})

        # ── HTML Console widget ───────────────────────────────────────────────
        self.console = widgets.HTML(
            value='',
            layout=widgets.Layout(
                height='480px',
                width='100%',
                overflow_y='scroll',
                border='1px solid #ccc',
                background_color='#fff',
            ))

        # ── Tab assembly ──────────────────────────────────────────────────────
        self.sub_tabs = widgets.Tab()
        self.sub_tabs.children = [self._ui_soul_forge(), self._ui_cockpit()]
        self.sub_tabs.set_title(0, "✨ Souls")
        self.sub_tabs.set_title(1, "🚀 Cockpit")
        self.sub_tabs.observe(self._on_sub_tab_change, names='selected_index')
        self._refresh_cockpit()

    def wire_logging(self):
        if 'shield'  in self.r: self.r['shield'].set_log_fn(self._log)
        if 'gateway' in self.r: self.r['gateway'].set_log_fn(self._log)
        self._log("🔌 T111: Console logging wired into T102 + T115.")

    # ── HTML rendering ────────────────────────────────────────────────────────

    def _flush(self):
        wrapper_open  = (
            '<div style="font-family:Arial,sans-serif;padding:8px;'
            'background:#fff;min-height:100%;">'
        )
        wrapper_close = '</div>'
        self.console.value = wrapper_open + ''.join(self._cards) + wrapper_close

    def _append_card(self, html: str):
        self._cards.append(html)
        if len(self._cards) > self.MAX_CARDS:
            self._cards = self._cards[-self.MAX_CARDS:]
        self._flush()

    def _log(self, message: str):
        safe = (str(message)
                .replace('&', '&amp;')
                .replace('<', '&lt;')
                .replace('>', '&gt;')
                .replace('\n', '<br>'))
        html = (
            f'<div style="font-size:12px;color:#666;font-family:monospace;'
            f'padding:2px 6px;line-height:1.6;">{safe}</div>'
        )
        self._append_card(html)

    def _render_maya_reflection(self, reflection_text: str):
        """
        BL-U08: Render Maya's closing reflection as a distinct indigo card.
        Strips the [MAYA — CLOSING REFLECTION]: prefix — card header carries that.
        """
        text = reflection_text
        prefix = "[MAYA — CLOSING REFLECTION]:"
        if text.startswith(prefix):
            text = text[len(prefix):].strip()

        def esc(s):
            return (s.replace('&', '&amp;')
                     .replace('<', '&lt;')
                     .replace('>', '&gt;')
                     .replace('\n', '<br>'))

        card = (
            '<div style="border:1px solid #5c6bc0;border-radius:8px;'
            'margin:10px 0;overflow:hidden;'
            'box-shadow:0 1px 4px rgba(92,107,192,0.3);">'
            '<div style="background:#3949ab;color:white;padding:8px 12px;'
            'font-size:14px;font-weight:bold;">🌌 Maya — Closing Reflection</div>'
            f'<div style="background:#e8eaf6;padding:12px 14px;font-size:13px;'
            f'color:#283593;font-style:italic;line-height:1.7;">'
            f'{esc(text)}</div>'
            '</div>'
        )
        self._append_card(card)

    def _render_turn(self, speaker: str, turn_text: str,
                     turn_count: int, urge: float, threshold: float,
                     margin: float, events: list):
        """Render one character turn as a visual card."""
        layers  = self._parse_turn_layers(turn_text)
        thought = layers['thought']
        speech  = layers['speech']

        def esc(s):
            return (s.replace('&', '&amp;')
                     .replace('<', '&lt;')
                     .replace('>', '&gt;')
                     .replace('\n', '<br>'))

        thought_html = ''
        if thought:
            thought_html = (
                f'<div style="background:#f2f2f2;padding:8px 12px;font-size:13px;'
                f'color:#444;border-top:1px solid #e0e0e0;">'
                f'💭 {esc(thought)}</div>'
            )

        event_labels = list({e[0] for e in events if e[0] != 'AMBIENT'})
        event_html   = ''
        if event_labels:
            labels_str = ' &nbsp; '.join(
                f'🔔 <strong>{esc(ev)}</strong>' for ev in event_labels
            )
            event_html = f'&nbsp;&nbsp;&nbsp; {labels_str}'
        meta_html = (
            f'<div style="padding:5px 12px;font-size:12px;color:#666;'
            f'border-top:1px solid #eee;">'
            f'⚡ Urge {urge} &middot; Threshold {threshold} &middot; '
            f'Margin {margin}{event_html}</div>'
        )

        speech_html = ''
        # BL-E07: passive turn detection ─────────────────────────────────────
        _passive_markers = ('*observes in silence*',)
        _is_passive_turn = speech and speech.strip() in _passive_markers
        if _is_passive_turn:
            speech_html = ('<div style="padding:6px 12px;font-size:12px;'
                           'color:#557799;font-style:italic;border-top:1px solid #eee;">'
                           '👁 observing silently</div>')
        elif speech:
            segments = self._split_speech(speech)
            # BL-U07: ensure at least one verbal segment ────────────────────
            if not any(k == 'verbal' for k, _ in segments):
                segments.append(('verbal', '--'))
            parts_html   = []
            first_verbal = True
            for kind, seg_text in segments:
                if kind == 'verbal':
                    prefix       = '💬 ' if first_verbal else ''
                    first_verbal = False
                    parts_html.append(
                        f'<div style="padding:8px 12px;font-size:14px;'
                        f'font-weight:bold;color:#111;border-top:1px solid #eee;">'
                        f'{prefix}{esc(seg_text)}</div>'
                    )
                else:
                    parts_html.append(
                        f'<div style="background:#f2f2f2;padding:6px 12px;'
                        f'font-size:13px;color:#555;border-top:1px solid #e0e0e0;">'
                        f'{esc(seg_text)}</div>'
                    )
            speech_html = ''.join(parts_html)
        # end BL-E07 / BL-U07 speech block ───────────────────────────────────

        card = (
            f'<div style="border:1px solid #ccc;border-radius:8px;'
            f'margin:10px 0;overflow:hidden;'
            f'box-shadow:0 1px 3px rgba(0,0,0,0.08);">'
            f'<div style="font-size:11px;color:#999;padding:3px 12px;'
            f'background:#fafafa;">Turn {turn_count}</div>'
            f'<div style="background:#1a5f7a;color:white;padding:8px 12px;'
            f'font-size:14px;font-weight:bold;">{esc(speaker)}</div>'
            f'{thought_html}'
            f'{meta_html}'
            f'{speech_html}'
            f'</div>'
        )
        self._append_card(card)

    def _split_speech(self, text: str) -> list:
        """Split speech into verbal (quoted) and action segments."""
        import re as _re
        segments = []
        pattern  = _re.compile(r'("(?:[^"\\]|\\.)*")')
        parts    = pattern.split(text)
        for part in parts:
            part = part.strip()
            if not part:
                continue
            if part.startswith('"') and part.endswith('"'):
                segments.append(('verbal', part))
            else:
                clean = _re.sub(r'\*([^*]+)\*', r'\1', part).strip()
                if clean:
                    segments.append(('action', clean))
        if not segments and text.strip():
            clean = _re.sub(r'\*([^*]+)\*', r'\1', text).strip()
            segments.append(('action', clean))
        return segments

    def _parse_turn_layers(self, turn_text: str) -> dict:
        layers  = {'thought': '', 'urge': '', 'speech': ''}
        pattern = re.compile(
            r'\[(Thought|Urge|Speech)\]\s*:?\s*(.*?)(?=\n?\[(?:Thought|Urge|Speech)\]|$)',
            re.IGNORECASE | re.DOTALL
        )
        for match in pattern.finditer(turn_text):
            key     = match.group(1).lower()
            content = match.group(2).strip()
            if key in layers:
                layers[key] = content
        if not any(layers.values()):
            layers['speech'] = turn_text.strip()
        return layers

    # ── Soul Forge ────────────────────────────────────────────────────────────

    def _ui_soul_forge(self):
        self.add_btn.on_click(self._add_row)
        self.save_btn.on_click(self._save_souls)
        return widgets.VBox([
            self.soul_canvas,
            widgets.HBox([self.add_btn, self.save_btn]),
            self.soul_status
        ])

    def _add_row(self, _):
        row = widgets.HBox([
            widgets.Text(placeholder="Name"),
            widgets.Text(placeholder="Identity"),
            widgets.Text(placeholder="Tone")
        ])
        self.soul_rows.append(row)
        self.soul_canvas.children = list(self.soul_canvas.children) + [row]

    def _save_souls(self, _):
        souls = self.r['state'].load_souls()
        for row in self.soul_rows:
            name, ident, tone = [c.value for c in row.children]
            if name:
                souls[name] = {"identity": ident, "mood": "Neutral", "tone": tone,
                               "dossier": ident, "last_utterances": []}
        self.r['state'].save_souls(souls)
        self.soul_rows = []
        self.soul_canvas.children = []
        self.soul_status.value = "Status: Souls born into Noah's Ark."
        self._refresh_cockpit()

    # ── Cockpit ───────────────────────────────────────────────────────────────

    def _ui_cockpit(self):
        self.start_btn.on_click(self._start_sim)
        self.kill_btn.on_click(self._kill_sim)
        self.new_scene_btn.on_click(self._new_scene)
        self.reset_ark_btn.on_click(self._reset_ark)
        self.pause_btn.on_click(self._toggle_pause)
        self.bft_btn.on_click(self._run_bft)
        self.wizard_btn.on_click(self._toggle_wizard)
        self.wizard_gen.on_click(self._run_wizard)
        self.inject_btn.on_click(self._inject_event)
        return widgets.VBox([
            widgets.VBox([
                widgets.Label("🌍 Environment:"),
                self.env_box,
            ], layout={'width': '100%', 'margin': '0 0 6px 0'}),
            widgets.VBox([
                widgets.Label("🎯 Scene Context (shared):"),
                self.obj_box,
            ], layout={'width': '100%', 'margin': '0 0 6px 0'}),
            widgets.VBox([
                widgets.Label("👥 Mission Personnel:"),
                self.cockpit_scroll_container,
            ], layout={'width': '100%', 'margin': '0 0 6px 0'}),
            widgets.HBox([
                self.new_scene_btn, self.start_btn, self.pause_btn,
                self.reset_ark_btn, self.kill_btn, self.bft_btn,
                self.pace_dropdown, self.scene_type_dd,
            ], layout={'margin': '4px 0', 'flex_wrap': 'wrap'}),
            widgets.HBox([
                self.wizard_btn, self.wizard_gen, self.wizard_status,
            ], layout={'margin': '2px 0'}),
            self.wizard_input,
            widgets.VBox([
                widgets.HTML('<div style="font-weight:600;color:#333;font-size:12px;'
                             'padding:3px 0;border-bottom:1px solid #e0e0e0;margin-bottom:3px;">'
                             '💉 Live Scene Injection</div>'),
                widgets.HBox([self.injection_box, self.inject_btn],
                             layout={'width': '99%'}),
            ], layout={'margin': '4px 0'}),
            self.console,
        ], layout={'width': '100%'})

    def _on_sub_tab_change(self, change):
        if change['new'] == 1:
            self._refresh_cockpit()

    def _refresh_cockpit(self):
        souls  = self.r['state'].load_souls()
        header = widgets.HBox([
            widgets.Label("Character",
                          layout={'width': '160px', 'font_weight': 'bold'}),
            widgets.Label("Status",
                          layout={'width': '110px', 'font_weight': 'bold'}),
            widgets.Label("Scene Role",
                          layout={'width': '220px', 'font_weight': 'bold'}),
            widgets.Label("Character Briefing (private knowledge)",
                          layout={'flex': '1 1 auto', 'font_weight': 'bold'}),
        ], layout={
            'width':         '99%',
            'border_bottom': '1px solid #555',
            'padding':       '2px 4px',
            'overflow':      'visible',
        })

        rows = [header]
        self.active_config = {}

        for name in souls:
            dd = widgets.Dropdown(
                options=['Active', 'Passive', 'No'], value='No',
                layout={'width': '105px'},
                style={'description_width': '0px'})
            role_text = widgets.Text(
                placeholder="e.g. Coordinator / Candidate",
                layout={'width': '215px', 'min_width': '0'})
            briefing_text = widgets.Text(
                placeholder="What this character knows at scene start (private)",
                layout={'flex': '1 1 auto', 'min_width': '0'})
            row = widgets.HBox(
                [widgets.Label(name, layout={'width': '160px'}),
                 dd, role_text, briefing_text],
                layout={'width': '99%', 'padding': '3px 4px', 'overflow': 'visible'})
            self.active_config[name] = {'status': dd, 'role': role_text,
                                        'briefing': briefing_text}
            rows.append(row)

        self.cockpit_canvas.children = rows

    def _on_scene_type_change(self, change):
        self.scene_type_value = change['new']

    def _toggle_wizard(self, _):
        visible = self.wizard_input.layout.display == 'none'
        self.wizard_input.layout.display = '' if visible else 'none'
        self.wizard_gen.layout.display   = '' if visible else 'none'

    def _run_wizard(self, _):
        desc = self.wizard_input.value.strip()
        if not desc:
            self.wizard_status.value = '⚠️ Enter a scene description first.'
            return
        if 'wizard' not in self.r:
            self.wizard_status.value = '⚠️ T119 SceneWizard not in registry.'
            return
        self.wizard_status.value = '✨ Generating...'
        try:
            active = [n for n, cfg in self.active_config.items()
                      if cfg['status'].value != 'No']
            result = self.r['wizard'].draft_scene(desc, active)
            if result:
                self.env_box.value = result.get('environment', '')
                self.obj_box.value = result.get('scene_context', '')
                roles = result.get('roles', {})
                for name, cfg in self.active_config.items():
                    if name in roles and cfg.get('role'):
                        cfg['role'].value = roles[name]
                self.wizard_status.value = '✅ Fields populated — review and edit before starting.'
            else:
                self.wizard_status.value = '⚠️ Wizard returned empty result.'
        except Exception as e:
            self.wizard_status.value = f'⚠️ Wizard error: {e}'

    def _inject_event(self, _):
        text = self.injection_box.value.strip()
        if not text:
            return
        event_str = f"[SCENE_EVENT]: {text}"
        try:
            self.r['state'].state['global_history'].append(event_str)
            self._log(f"💉 Injected: {event_str}")
            self.injection_box.value = ''
        except Exception as e:
            self._log(f"⚠️ Injection error: {e}")

    def _on_scene_type_change(self, change):
        self.scene_type_value = change['new']

    def _toggle_wizard(self, _):
        visible = self.wizard_input.layout.display == 'none'
        self.wizard_input.layout.display = '' if visible else 'none'
        self.wizard_gen.layout.display   = '' if visible else 'none'

    def _run_wizard(self, _):
        desc = self.wizard_input.value.strip()
        if not desc:
            self.wizard_status.value = '⚠️ Enter a scene description first.'
            return
        if 'wizard' not in self.r:
            self.wizard_status.value = '⚠️ T119 SceneWizard not in registry.'
            return
        self.wizard_status.value = '✨ Generating...'
        try:
            active = [n for n, cfg in self.active_config.items()
                      if cfg['status'].value != 'No']
            result = self.r['wizard'].draft_scene(desc, active)
            if result:
                self.env_box.value = result.get('environment', '')
                self.obj_box.value = result.get('scene_context', '')
                roles = result.get('roles', {})
                for name, cfg in self.active_config.items():
                    if name in roles and cfg.get('role'):
                        cfg['role'].value = roles[name]
                self.wizard_status.value = '✅ Fields populated — review and edit before starting.'
            else:
                self.wizard_status.value = '⚠️ Wizard returned empty result.'
        except Exception as e:
            self.wizard_status.value = f'⚠️ Wizard error: {e}'

    def _inject_event(self, _):
        text = self.injection_box.value.strip()
        if not text:
            return
        event_str = f"[SCENE_EVENT]: {text}"
        try:
            self.r['state'].state['global_history'].append(event_str)
            self._log(f"💉 Injected: {event_str}")
            self.injection_box.value = ''
        except Exception as e:
            self._log(f"⚠️ Injection error: {e}")

    def _run_bft(self, _):
        """BL-I05: Run BFT SmokeTester from cockpit button."""
        if 'tester' not in self.r:
            self._log("⚠️ T116 SmokeTester not in registry.")
            return
        self._log("🧪 Running BFT SmokeTester...")
        try:
            result = self.r['tester'].run()
            status = "✅ BFT PASSED" if result else "❌ BFT FAILED"
            self._log(f"{status} — check notebook output for details.")
        except Exception as e:
            self._log(f"⚠️ BFT error: {e}")

    def get_render_box(self):
        return self.sub_tabs

    # ── Button Handlers ───────────────────────────────────────────────────────

    def _new_scene(self, _):
        if self.is_running:
            self._log("⚠️ Stop the simulation before starting a new scene.")
            return
        self.env_box.value = ""
        self.obj_box.value = ""
        self._log("🆕 New Scene: Environment and Scene Context cleared.")
        self._log("   History + Souls preserved. Set new Env/Context then ▶ Start.")

    def _toggle_pause(self, _):
        if not self.is_running:
            self._log("⚠️ Simulation is not running.")
            return
        self.pause_flag = not self.pause_flag
        if self.pause_flag:
            self.pause_btn.description = "▶ Resume"
            self._log("⏸ Scene paused.")
        else:
            self.pause_btn.description = "⏸ Pause"
            self._log("▶ Resuming scene...")

    def _reset_ark(self, _):
        if self.is_running:
            self._log("⚠️ Hard Kill first before resetting the Ark.")
            return
        self._log("🔄 Reset Ark initiated...")
        try:
            env     = self.env_box.value
            obj     = self.obj_box.value
            history = self.r['state'].state.get('global_history', [])
            if history and 'archiver' in self.r:
                archive_file = self.r['archiver'].archive(env, obj, history)
                self._log(f"📦 Safety archive: {archive_file}")
            self.r['state'].reset_scene()
            self._cards = []
            self._flush()
            self._log("✅ Ark reset. History + utterances cleared. Souls preserved.")
            self._log("   Set new Environment + Scene Context and ▶ Start.")
        except Exception as e:
            self._log(f"⚠️ Reset Ark error: {e}")

    def _start_sim(self, _):
        if not self.is_running:
            self.is_running  = True
            self.pause_flag  = False
            self.pause_btn.description = "⏸ Pause"
            self._cards = []
            self._flush()

            # BL-I06: Initialise SceneLedger for this scene ──────────────────
            scene_id = f"Scene_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
            if 'sentinel' in self.r:
                self.r['sentinel'].start_scene(scene_id)

            self._log("🚀 Noah's Ark OS: Scene started.")
            self._log(f"🎯 Scene Context: {self.obj_box.value[:120]}")
            self._log(f"⏱ Pace: {self.pace_dropdown.value}")
            self._log("─" * 60)
            active_at_start = [
                name for name, cfg in self.active_config.items()
                if cfg['status'].value == 'Active'
            ]
            if 'conductor' in self.r:
                cond = self.r['conductor']
                cond.reset_urge_state(active_at_start)
                cond.reset_threshold_state(active_at_start)
                cond.build_domain_map(active_at_start)
                thr_snapshot = {s: round(cond.threshold_state.get(s, 6.0), 1)
                                for s in active_at_start}
                # BL-U10: per-character urge display (threshold + 1.5) ─────────
                urge_snapshot = {s: round(cond.threshold_state.get(s, 6.0) + 1.5, 1)
                                 for s in active_at_start}
                self._log(f"⚡ {len(active_at_start)} souls initialised — "
                          f"Urges: {urge_snapshot} | Thresholds: {thr_snapshot}")
            self._prev_active_souls = []  # BL-E11: entry detection snapshot
            threading.Thread(target=self._run_simulation_loop, daemon=True).start()

    def _kill_sim(self, _):
        self.is_running = False
        self.pause_flag = False
        self.pause_btn.description = "⏸ Pause"
        self._log("⏹ Hard Kill — saving world state...")
        try:
            env     = self.env_box.value
            obj     = self.obj_box.value
            history = self.r['state'].state.get('global_history', [])
            self.r['state'].save_history()
            self._log(f"💾 global_history.json saved ({len(history)} entries).")
            if 'archiver' in self.r:
                af = self.r['archiver'].archive(env, obj, history)
                if af: self._log(f"📦 Scene archived → {af}")
            stats = self.r['sentinel'].get_totals()
            self._log(f"📊 Session cost: ${stats['usd']:.6f} | "
                      f"Tokens: Prompt {stats['p']} | Completion {stats['c']}")
            self._log("✅ Engine stopped.")
        except Exception as e:
            self._log(f"⚠️ Save error during kill: {e}")

    # ── Simulation Loop ───────────────────────────────────────────────────────

    def _run_simulation_loop(self):
        try:
            turn_count = 0
            while self.is_running:

                if self.pause_flag:
                    time.sleep(0.5)
                    continue

                active_souls = [
                    name for name, cfg in self.active_config.items()
                    if cfg['status'].value == 'Active'
                ]
                if not active_souls:
                    self._log("⚠️ No Active souls. Standing by...")
                    time.sleep(3)
                    continue

                # BL-E11: Mid-scene entry detection ──────────────────────
                if hasattr(self, '_prev_active_souls') and self._prev_active_souls:
                    new_entrants = [s for s in active_souls
                                    if s not in self._prev_active_souls]
                    for entrant in new_entrants:
                        ev = f"[SCENE_EVENT]: {entrant} has entered the room."
                        self.r['state'].log_event(ev)
                        self._log(f"🚪 {entrant} entered — SCENE_EVENT written to history.")
                self._prev_active_souls = list(active_souls)

                conductor = self.r['conductor']
                speaker   = conductor.select_next_speaker(active_souls)

                if speaker is None:
                    conductor.tick_all_souls(active_souls)
                    self._log("💭 Scene breathes...")
                    time.sleep(1.5)
                    continue

                while self.pause_flag and self.is_running:
                    time.sleep(0.5)
                if not self.is_running:
                    break

                turn_count += 1
                u  = round(conductor.urge_state.get(speaker, 7.0), 1)
                th = round(conductor.threshold_state.get(speaker, 6.0), 1)
                mg = round(u - th, 1)

                try:
                    turn_text, scene_end, new_urge = conductor.run_turn(speaker)
                    conductor.update_urge(speaker, new_urge)
                    events = conductor.post_turn(turn_text, speaker, active_souls)
                    conductor.tick_silent_souls(active_souls, speaker)
                    self._render_turn(speaker, turn_text,
                                      turn_count, u, th, mg, events)

                    if scene_end:
                        self._log("🎬 [[SCENE_END]] detected — scene complete.")
                        self.is_running = False
                        self._do_scene_end_archive()
                        break

                except Exception as e:
                    self._log(f"❌ ERROR in {speaker}'s turn: {e}")
                    self._log(traceback.format_exc())
                    self.is_running = False
                    break

                delay = self.get_pace_delay(turn_text or "")
                if delay > 0:
                    time.sleep(delay)

        except Exception as fatal_e:
            self._log(f"💥 FATAL THREAD CRASH: {fatal_e}")
            self._log(traceback.format_exc())
            self.is_running = False

    def get_pace_delay(self, last_response_text: str) -> float:
        mode       = self.pace_dropdown.value
        word_count = len(last_response_text.split()) if last_response_text else 0
        if   mode == 'Digital':   return 0.0
        elif mode == 'Human':     return 2.0 + (word_count * 0.02)
        elif mode == 'Cinematic': return 3.0 + (word_count * 0.04)
        return 2.0

    def _do_scene_end_archive(self):
        """
        BL-E01: Clean archive on [[SCENE_END]].
        BL-M01: Maya closing reflection.
        BL-U08: Maya reflection as indigo card.
        BL-I06: SceneLedger summary + pass token_log to archiver.
        BL-I07: Pass state_engine to archiver for boundary marker.
        """
        try:
            env     = self.env_box.value
            obj     = self.obj_box.value
            history = self.r['state'].state.get('global_history', [])
            self.r['state'].save_history()
            self._log(f"💾 global_history.json saved ({len(history)} entries).")

            # BL-M01 + BL-U08: Maya closing reflection ─────────────────────
            maya_reflection = None
            if 'maya' in self.r:
                try:
                    self._log("🌌 Consulting Maya for closing reflection...")
                    # BL-Q04: build turns_per_character ─────────────────
                    tpc = {}
                    if 'sentinel' in self.r:
                        sd = self.r['sentinel'].get_scene_dict()
                        tpc = {n: ct.get('turns', 0)
                               for n, ct in sd.get('char_totals', {}).items()}
                    maya_reflection = self.r['maya'].close_scene(
                        env, obj, history, turns_per_character=tpc)
                    if maya_reflection:
                        self._render_maya_reflection(maya_reflection)  # BL-U08
                except Exception as me:
                    self._log(f"⚠️ Maya closing thought failed (non-blocking): {me}")

            # BL-I06: Scene token summary ───────────────────────────────────
            token_log = {}
            if 'sentinel' in self.r:
                scene_lines = self.r['sentinel'].get_scene_summary()
                if scene_lines:
                    self._log("📊 " + scene_lines[0])   # Scene ID line
                    for line in scene_lines[1:]:
                        self._log(line)
                token_log = self.r['sentinel'].get_scene_dict()

            # Session total ────────────────────────────────────────────────
            stats = self.r['sentinel'].get_totals()
            self._log(f"📊 Session total: ${stats['usd']:.6f} | "
                      f"P:{stats['p']} C:{stats['c']}")

            # BL-I07 + BL-I06: Archive with token_log + state_engine ───────
            if 'archiver' in self.r:
                af = self.r['archiver'].archive(
                    env, obj, history,
                    maya_reflection=maya_reflection,
                    token_log=token_log,
                    state_engine=self.r.get('state')
                )
                if af: self._log(f"📦 Scene archived → {af}")

            self._log("✅ World state saved. Engine fully stopped.")
        except Exception as e:
            self._log(f"⚠️ Scene end archive error: {e}")

# ------------------------------------------
# IPO CHECK:
# Input:   Button clicks, Soul Forge entries, Cockpit config
# Process: Thread mgmt, pause_flag, _render_turn() HTML card output,
#          _render_maya_reflection() indigo card (BL-U08),
#          sentinel.start_scene() (BL-I06), T114 delegation,
#          Maya reflection, T109 archive with token_log + state_engine
# Output:  HTML console (turn cards + system log lines), archive triggers
#
# New in v5.10.2:
#   BL-U07: action-only speech → ('verbal', '--') placeholder injected
#   BL-U08: _render_maya_reflection() — indigo card for Maya reflection
#   BL-I06: start_scene() on _start_sim(), scene summary on archive
# ------------------------------------------


In [41]:
# ==========================================
# T112 : UI_ORACLE_TAB
# ==========================================
# VERSION: 5.2.0 | STATUS: Stable
# ROLE: Oracle Gateway with User Query Input
# Version Remarks: Unchanged from v1.8.0.
# ------------------------------------------

import ipywidgets as widgets

class UIOracleTab:
    def __init__(self):
        self.r = None
        self.output_area = widgets.Textarea(
            value='', disabled=True,
            placeholder='Oracle responses will appear here...',
            layout=widgets.Layout(
                height='350px', width='100%',
                overflow_y='scroll', border='1px solid #555',
                font_family='monospace', font_size='12px'))
        self.input_text = widgets.Text(
            placeholder="Ask your Oracle...", layout={'width': '70%'})
        self.submit_btn = widgets.Button(
            description="👁️ Consult Oracle", button_style='info')
        self.clear_btn  = widgets.Button(
            description="🗑 Clear", button_style='warning', layout={'width': '80px'})

    def _append(self, message: str):
        current = self.output_area.value
        self.output_area.value = (current + "\n" + str(message)).lstrip("\n")

    def on_submit(self, _):
        query = self.input_text.value.strip()
        if not query:
            return
        self.input_text.value = ""
        self._append(f"\n> {query}")
        self._append("👁️ Consulting the Oracle...")
        oracle_logic = self.r.get('oracle')
        if oracle_logic:
            oracle_text = oracle_logic.ask(query)
        else:
            response_package = self.r['brain'].generate_urge("Oracle", query)
            oracle_text = response_package[0] if response_package else "No response."
        self._append(f"\n🔮 Oracle:\n{oracle_text}")
        stats = self.r['sentinel'].get_totals()
        self._append(f"\n── Session Cost: ${stats['usd']:.6f} | "
                     f"Tokens P:{stats['p']} C:{stats['c']} ──")
        self._append("-" * 40)

    def _clear_output(self, _):
        self.output_area.value = ""

    def get_render_box(self):
        self.submit_btn.on_click(self.on_submit)
        self.clear_btn.on_click(self._clear_output)
        return widgets.VBox([
            self.output_area,
            widgets.HBox([self.input_text, self.submit_btn, self.clear_btn])
        ])

# ------------------------------------------
# IPO CHECK:
# Input:   User query string
# Process: Route to T110 → append to Textarea
# Output:  Oracle response in scrollable Textarea
# ------------------------------------------


In [42]:
# ==========================================
# T113 : UI_METRICS
# ==========================================
# VERSION: 5.1.2 | STATUS: Stable
# ROLE: Real-Time Financial Telemetry
# Version Remarks: Unchanged from v1.8.0.
# ------------------------------------------

import ipywidgets as widgets

class UIMetrics:
    def __init__(self):
        self.cost_label      = widgets.Label(value="Session Cost (USD): $0.000000")
        self.token_info      = widgets.HTML(value="<b>Tokens:</b> P: 0 | C: 0")
        self.sentinel_handle = None

    def refresh_data(self):
        if self.sentinel_handle:
            stats = self.sentinel_handle.get_totals()
            self.cost_label.value  = f"Session Cost (USD): ${stats['usd']:.6f}"
            self.token_info.value  = f"<b>Tokens:</b> Prompt: {stats['p']} | Completion: {stats['c']}"

    def get_render_box(self):
        return widgets.VBox([self.cost_label, self.token_info])

    def display_metrics(self):
        display(self.get_render_box())

# ------------------------------------------
# IPO CHECK:
# Input:   Token ledger data from T108
# Process: Pull totals → format → update widgets
# Output:  Live cost + token display
# ------------------------------------------


In [43]:
# ==========================================
# T114 : SIM_CONDUCTOR
# ==========================================
# VERSION: 5.5.5 | STATUS: Evolved
# ROLE: Orchestration, Urge + Threshold Selection, Event Classification
# Version Remarks: v5.6.0 — BL-E07 Sprint 6.
#   reset_urge_state() changed from flat 7.0 to threshold + 1.5 per character.
#   High-threshold tones (Calculated 7.0, Wise 7.5, Enigmatic 8.0) previously
#   started at zero or negative margin — requiring 4-5 passive ticks before
#   first selection. In short scenes they were structurally silenced.
#   Fix: urge = TONE_THRESHOLD + 1.5 gives every character margin 1.5 at
#   scene start. Differentiation comes from event response and tone increment.
#   Requires reset_threshold_state() called first — already the case in T111.
#
#   v5.5.2 — BL-E08 Sprint 3.
#
#   v5.5.1 — Sprint 2 extended. Three fixes:
#   FIX-1: Tiebreak randomisation — tied margins pick randomly, not dict-order.
#   FIX-2: CONTRADICTION_KW tightened — "but" removed; requires strong signal.
#   FIX-3: THRESHOLD_FLOOR raised 3.0→5.0; TONE_THRESHOLD Anxious 5.0→5.5.
#
#   Original v5.5.0 additions:
#
#   EXPRESSION THRESHOLD (float per character):
#     Urge and threshold are now separate values. Selection uses margin
#     (urge - threshold). Highest margin speaks. A character can be
#     highly activated but held back by a high threshold — creating the
#     "clearly wants to speak but hasn't yet" state.
#     Tone sets initial threshold. Events evolve it over the scene.
#
#   THREE-LAYER EVENT CLASSIFIER (post_turn):
#     After each turn, classify what kind of event just occurred and
#     apply urge + threshold deltas to all characters accordingly.
#     Layer 1: explicit name in utterance  → confidence 1.0
#     Layer 2: pronoun resolution          → confidence 0.8
#     Layer 3: domain keyword inference    → confidence 0.6
#     They/them: domain scan first → stakes check → ignore
#     Multiple events can fire from a single utterance.
# ------------------------------------------

import re
from collections import deque

class SimConductor:

    # ── Tone Tables ────────────────────────────────────────────────────────────
    TONE_INCREMENT = {
        "Anxious":      2, "Inquisitive":  2,
        "Neutral":      1, "Professional": 1,
        "Calculated":   1, "Wise":         1, "Enigmatic": 1,
    }
    DEFAULT_INCREMENT = 1

    # Initial expression threshold per tone --------------------------------
    # Low = speaks easily. High = holds back until compelled.
    TONE_THRESHOLD = {
        "Anxious":      5.5,   # blurts, low inhibition — raised to 5.5 (floor is 5.0)
        "Inquisitive":  5.5,   # asks freely
        "Neutral":      6.0,
        "Professional": 6.5,   # waits for right moment
        "Calculated":   7.0,   # speaks when certain
        "Wise":         7.5,   # speaks rarely
        "Enigmatic":    8.0,   # almost never volunteers
    }
    DEFAULT_THRESHOLD = 6.0
    THRESHOLD_FLOOR   = 5.0   # never fully locked out — raised from 3.0 to prevent collapse
    THRESHOLD_CEIL    = 9.0   # never fully silenced

    # ── Confidence Multipliers by Resolution Layer ─────────────────────────────
    L1 = 1.0   # explicit name
    L2 = 0.8   # pronoun resolved
    L3 = 0.6   # domain inferred
    LS = 0.5   # out-of-room stakes

    # ── Event Detection Keywords ───────────────────────────────────────────────
    CONTRADICTION_KW  = [
        "however", "actually", "disagree", "wrong",
        "not quite", "wouldn\'t say", "i don\'t think", "that\'s not",
        "i\'d argue", "i disagree", "not correct", "mistaken", "incorrect"
    ]
    # NOTE: "but" removed — fires on almost every polite sentence and tanks
    # thresholds to floor within 10 turns. Strong signals only.
    AGREEMENT_KW      = [
        "exactly", "agreed", "absolutely", "precisely",
        "indeed", "correct", "you\'re right", "i agree"
    ]
    ESCALATION_KW     = [
        "urgent", "critical", "danger", "alarm", "emergency",
        "fail", "crisis", "now", "immediately", "must"
    ]
    STAKES_KW         = [
        "board", "director", "committee", "decision",
        "select", "chosen", "panel", "review", "judge"
    ]
    STOPWORDS = {
        "the","a","an","and","or","but","in","of","to","is","are","was",
        "were","has","have","had","be","been","being","for","with","that",
        "this","it","at","by","from","on","as","into","their","they","we",
        "i","you","he","she","us","our","my","its","which","who","what",
        "when","where","how","all","just","not","do","did","will","would",
        "could","should","than","then","so","if","about","any","some","more"
    }

    def __init__(self, registry):
        self.r              = registry
        self.urge_state     = {}   # {actor: float}
        self.threshold_state= {}   # {actor: float}
        self.domain_map     = {}   # {keyword: [actor_names]}
        self._last_speaker  = None # track for pronoun "you" resolution
        self._recent_speakers = deque(maxlen=3)  # last 3 speakers

    # ══════════════════════════════════════════════════════════════════════════
    # SECTION 1: INITIALISATION
    # ══════════════════════════════════════════════════════════════════════════

    def reset_urge_state(self, active_souls):
        """BL-C02 + BL-N02: Initialise urge with scene type modifiers from T120."""
        self._last_speaker    = None
        self._recent_speakers = deque(maxlen=3)

        # BL-N02: apply T120 scene type modifiers if available ----------------
        scene_type = 'Social'
        role_map   = {}
        if 'ui_cmd' in self.r:
            ui = self.r['ui_cmd']
            scene_type = getattr(ui, 'scene_type_value', 'Social')
            role_map   = {
                name: cfg['role'].value.lower()
                for name, cfg in ui.active_config.items()
                if cfg.get('role')
            }

        orchestrator = self.r.get('orchestrator')
        if orchestrator:
            self.urge_state = orchestrator.apply_scene_type(
                active_souls, self.threshold_state, scene_type, role_map)
        else:
            self.urge_state = {
                s: self.threshold_state.get(s, self.DEFAULT_THRESHOLD) + 1.5
                for s in active_souls
            }
        # deque continuation below — do not remove this comment


    def reset_threshold_state(self, active_souls):
        """Initialise expression threshold from soul tone. Called on Start."""
        self.threshold_state = {}
        for soul in active_souls:
            try:
                data = self.r['state'].get_actor_data(soul)
                tone = data.get('tone', 'Neutral')
                self.threshold_state[soul] = self.TONE_THRESHOLD.get(
                    tone, self.DEFAULT_THRESHOLD)
            except Exception:
                self.threshold_state[soul] = self.DEFAULT_THRESHOLD

    def build_domain_map(self, active_souls):
        """
        Build keyword → [actors] map from soul domain_keywords field.
        Falls back to dossier word scan if domain_keywords not present.
        Called on Start so the map is ready for the whole scene.
        """
        self.domain_map = {}
        for soul in active_souls:
            try:
                data = self.r['state'].get_actor_data(soul)
                keywords = data.get('domain_keywords', [])
                if not keywords:
                    # Fallback: scan dossier, filter stopwords
                    dossier = data.get('dossier', '')
                    words   = re.findall(r'[a-z]+', dossier.lower())
                    keywords = [w for w in words
                                if w not in self.STOPWORDS and len(w) > 3]
                for kw in keywords:
                    kw = kw.lower()
                    self.domain_map.setdefault(kw, [])
                    if soul not in self.domain_map[kw]:
                        self.domain_map[kw].append(soul)
            except Exception:
                pass

    # ══════════════════════════════════════════════════════════════════════════
    # SECTION 2: SPEAKER SELECTION
    # ══════════════════════════════════════════════════════════════════════════

    def select_next_speaker(self, active_souls):
        """
        Select character with highest (urge - threshold) margin.
        Margin must be positive — character must exceed their own threshold.
        Returns actor name or None if nobody is above their threshold.
        """
        for soul in active_souls:
            self.urge_state.setdefault(soul, 7.0)
            self.threshold_state.setdefault(soul, self.DEFAULT_THRESHOLD)

        margins = {
            s: self.urge_state[s] - self.threshold_state[s]
            for s in active_souls
        }
        eligible = {s: m for s, m in margins.items() if m > 0}

        if not eligible:
            return None  # Scene breathes

        # Tiebreak: randomise among candidates within 0.5 of the top margin --
        # Prevents dict-order bias (Valentina always losing to Abdul/Rebecca/Vel)
        import random
        top_margin = max(eligible.values())
        tied = [s for s, m in eligible.items() if top_margin - m <= 0.5]
        return random.choice(tied)

    # ══════════════════════════════════════════════════════════════════════════
    # SECTION 3: URGE + THRESHOLD EVOLUTION
    # ══════════════════════════════════════════════════════════════════════════

    def _get_increment(self, actor):
        try:
            tone = self.r['state'].get_actor_data(actor).get('tone', 'Neutral')
            return self.TONE_INCREMENT.get(tone, self.DEFAULT_INCREMENT)
        except Exception:
            return self.DEFAULT_INCREMENT

    def _clamp_urge(self, v):
        return max(1.0, min(10.0, v))

    def _clamp_threshold(self, v):
        return max(self.THRESHOLD_FLOOR, min(self.THRESHOLD_CEIL, v))

    def update_urge(self, actor, new_urge):
        """Store Urge from model response. Speaker's own output is ground truth."""
        self.urge_state[actor] = self._clamp_urge(float(new_urge))

    def tick_silent_souls(self, active_souls, speaker):
        """
        Passive increment for characters who did not speak this turn.
        Rate is tone-derived. This is additive to any event deltas.
        """
        for soul in active_souls:
            if soul != speaker:
                inc = self._get_increment(soul)
                self.urge_state[soul] = self._clamp_urge(
                    self.urge_state.get(soul, 7.0) + inc)

    def tick_all_souls(self, active_souls):
        """Scene breathes — nobody spoke. All gain passive increment."""
        for soul in active_souls:
            inc = self._get_increment(soul)
            self.urge_state[soul] = self._clamp_urge(
                self.urge_state.get(soul, 7.0) + inc)

    # ══════════════════════════════════════════════════════════════════════════
    # SECTION 4: EVENT CLASSIFICATION (THREE-LAYER)
    # ══════════════════════════════════════════════════════════════════════════

    def _domain_targets(self, utterance_lower, active_souls, exclude=None):
        """
        Layer 3: find active characters whose domain keywords appear in utterance.
        Returns list of (actor, count) sorted by keyword hit count descending.
        """
        hits = {}
        for kw, owners in self.domain_map.items():
            if kw in utterance_lower:
                for owner in owners:
                    if owner in active_souls and owner != exclude:
                        hits[owner] = hits.get(owner, 0) + 1
        return sorted(hits.keys(), key=lambda x: hits[x], reverse=True)

    def _sentiment_near_name(self, utterance_lower, name_lower):
        """
        Extract a sentiment window of ±60 chars around the name.
        Returns (is_contradiction, is_agreement).
        """
        idx = utterance_lower.find(name_lower)
        if idx < 0:
            return False, False
        window = utterance_lower[max(0, idx-60): idx+len(name_lower)+60]
        is_c = any(w in window for w in self.CONTRADICTION_KW)
        is_a = any(w in window for w in self.AGREEMENT_KW)
        return is_c, is_a

    def classify_event(self, utterance, speaker, active_souls):
        """
        Classify what kind of conversational event just occurred.
        Returns list of (event_type, target, confidence_mult).
        Multiple events can coexist in one utterance.

        event_type values:
          DIRECT_QUESTION  — question aimed at specific actor
          OPEN_QUESTION    — question to the group
          CONTRADICTION    — disagreement with specific actor or group
          AGREEMENT        — agreement with specific actor or group
          DOMAIN_MENTION   — implicit reference to someone's domain
          ESCALATION_WIN   — speaker escalates, is the initiator
          ESCALATION_LOSS  — named actor is the target of escalation
          EMOTIONAL_ESC    — general emotional escalation, no clear target
          STAKES           — out-of-room authority referenced (they/board/etc)
          AMBIENT          — no classifiable event
        """
        utt = utterance.lower()
        events = []

        # ── Named actors present in utterance (Layer 1) ───────────────────
        named = [s for s in active_souls
                 if s.lower() in utt and s != speaker]

        has_question    = '?' in utterance
        has_contra      = any(w in utt for w in self.CONTRADICTION_KW)
        has_agree       = any(w in utt for w in self.AGREEMENT_KW)
        has_escalation  = any(w in utt for w in self.ESCALATION_KW)

        # ── Questions ─────────────────────────────────────────────────────
        if has_question:
            if named:
                for actor in named:
                    events.append(('DIRECT_QUESTION', actor, self.L1))
            elif 'you' in utt.split() or utt.startswith('you'):
                target = self._last_speaker
                if target and target in active_souls:
                    events.append(('DIRECT_QUESTION', target, self.L2))
                else:
                    events.append(('OPEN_QUESTION', 'ALL', self.L2))
            elif ' we ' in utt or ' us ' in utt:
                events.append(('OPEN_QUESTION', 'ALL', self.L2))
            else:
                events.append(('OPEN_QUESTION', 'ALL', 1.0))

        # ── Contradiction / Agreement ──────────────────────────────────────
        if has_contra or has_agree:
            if named:
                # Per-name sentiment window — one actor can be agreed with
                # and another contradicted in the same utterance
                for actor in named:
                    is_c, is_a = self._sentiment_near_name(utt, actor.lower())
                    if is_c:
                        events.append(('CONTRADICTION', actor, self.L1))
                    if is_a:
                        events.append(('AGREEMENT',    actor, self.L1))
                    # If sentiment window is ambiguous, check global signal
                    if not is_c and not is_a:
                        if has_contra:
                            events.append(('CONTRADICTION', actor, self.L1 * 0.5))
                        if has_agree:
                            events.append(('AGREEMENT',    actor, self.L1 * 0.5))

            elif 'you' in utt.split() or utt.startswith('you'):
                target = self._last_speaker
                if target and target in active_souls:
                    if has_contra:
                        events.append(('CONTRADICTION', target, self.L2))
                    if has_agree:
                        events.append(('AGREEMENT',    target, self.L2))

            elif ' we ' in utt or ' us ' in utt:
                if has_agree:
                    events.append(('AGREEMENT', 'ALL', self.L2))
                if has_contra:
                    events.append(('CONTRADICTION', 'ALL', self.L2 * 0.6))

            else:
                # Layer 3: domain inference
                d_targets = self._domain_targets(utt, active_souls, speaker)
                for actor in d_targets[:2]:   # cap at 2 inferred targets
                    if has_contra:
                        events.append(('CONTRADICTION', actor, self.L3))
                    if has_agree:
                        events.append(('AGREEMENT',    actor, self.L3))

        # ── Domain Mention (independent — only if not already targeted) ───
        d_targets = self._domain_targets(utt, active_souls, speaker)
        already_targeted = {e[1] for e in events}
        for actor in d_targets[:2]:
            if actor not in already_targeted:
                events.append(('DOMAIN_MENTION', actor, self.L3))

        # ── Emotional Escalation ──────────────────────────────────────────
        if has_escalation:
            if named:
                # Named actor = LOSS target. Speaker = WIN.
                for actor in named:
                    events.append(('ESCALATION_LOSS', actor, self.L1))
                events.append(('ESCALATION_WIN', speaker, 1.0))

            elif 'they' in utt.split() or 'them' in utt.split():
                # they/them → domain scan first
                d_targets = self._domain_targets(utt, active_souls, speaker)
                if d_targets:
                    for actor in d_targets[:2]:
                        events.append(('ESCALATION_LOSS', actor, self.L3))
                    events.append(('ESCALATION_WIN', speaker, 1.0))
                else:
                    # Stakes check — out-of-room authority
                    has_stakes = any(w in utt for w in self.STAKES_KW)
                    if has_stakes:
                        events.append(('STAKES', 'ALL', self.LS))
                    # else: true ambient they/them — ignore
            else:
                events.append(('EMOTIONAL_ESC', 'ALL', 1.0))

        # ── Ambient fallback ──────────────────────────────────────────────
        if not events:
            events.append(('AMBIENT', None, 1.0))

        return events

    # ══════════════════════════════════════════════════════════════════════════
    # SECTION 5: EVENT APPLICATION (URGE + THRESHOLD DELTAS)
    # ══════════════════════════════════════════════════════════════════════════

    def apply_event(self, events, speaker, active_souls):
        """
        Apply urge and threshold deltas to all characters based on classified events.
        Bystander proximity: recent_speakers deque determines closeness to conflict.
        """
        for event_type, target, conf in events:

            if event_type == 'DIRECT_QUESTION':
                if target in active_souls:
                    self.urge_state[target]      = self._clamp_urge(
                        self.urge_state.get(target, 7.0) + 3.0 * conf)
                    self.threshold_state[target]  = self._clamp_threshold(
                        self.threshold_state.get(target, 6.0) - 0.5 * conf)

            elif event_type == 'OPEN_QUESTION':
                for soul in active_souls:
                    if soul != speaker:
                        self.urge_state[soul] = self._clamp_urge(
                            self.urge_state.get(soul, 7.0) + 1.5 * conf)

            elif event_type == 'CONTRADICTION':
                if target == 'ALL':
                    for soul in active_souls:
                        if soul != speaker:
                            self.urge_state[soul] = self._clamp_urge(
                                self.urge_state.get(soul, 7.0) + 1.0 * conf)
                            self.threshold_state[soul] = self._clamp_threshold(
                                self.threshold_state.get(soul, 6.0) - 0.2 * conf)
                elif target in active_souls:
                    # Target: compelled to defend
                    self.urge_state[target]      = self._clamp_urge(
                        self.urge_state.get(target, 7.0) + 2.0 * conf)
                    self.threshold_state[target]  = self._clamp_threshold(
                        self.threshold_state.get(target, 6.0) - 0.4 * conf)
                    # Bystanders: tension observer spike
                    for soul in active_souls:
                        if soul != speaker and soul != target:
                            prox = 0.7 if soul in self._recent_speakers else 0.4
                            self.urge_state[soul] = self._clamp_urge(
                                self.urge_state.get(soul, 7.0) + 0.5 * conf * prox)

            elif event_type == 'AGREEMENT':
                if target == 'ALL':
                    for soul in active_souls:
                        self.urge_state[soul] = self._clamp_urge(
                            self.urge_state.get(soul, 7.0) - 0.3 * conf)
                elif target in active_souls:
                    # Target: validated — slight step back
                    self.urge_state[target]      = self._clamp_urge(
                        self.urge_state.get(target, 7.0) - 0.5 * conf)
                    self.threshold_state[target]  = self._clamp_threshold(
                        self.threshold_state.get(target, 6.0) + 0.2 * conf)

            elif event_type == 'DOMAIN_MENTION':
                if target in active_souls:
                    self.urge_state[target]      = self._clamp_urge(
                        self.urge_state.get(target, 7.0) + 2.5 * conf)
                    self.threshold_state[target]  = self._clamp_threshold(
                        self.threshold_state.get(target, 6.0) - 0.3 * conf)

            elif event_type == 'ESCALATION_LOSS':
                if target in active_souls:
                    self.urge_state[target]      = self._clamp_urge(
                        self.urge_state.get(target, 7.0) + 3.0 * conf)
                    self.threshold_state[target]  = self._clamp_threshold(
                        self.threshold_state.get(target, 6.0) - 0.5 * conf)

            elif event_type == 'ESCALATION_WIN':
                if target in active_souls:
                    self.urge_state[target] = self._clamp_urge(
                        self.urge_state.get(target, 7.0) + 1.0 * conf)
                    self.threshold_state[target] = self._clamp_threshold(
                        self.threshold_state.get(target, 6.0) - 0.3 * conf)
                # Bystanders: proximity-weighted
                for soul in active_souls:
                    if soul != target:
                        prox = 1.0 if soul in self._recent_speakers else 0.5
                        self.urge_state[soul] = self._clamp_urge(
                            self.urge_state.get(soul, 7.0) + 0.5 * conf * prox)

            elif event_type == 'EMOTIONAL_ESC':
                # General escalation — everyone activates
                for soul in active_souls:
                    self.urge_state[soul]      = self._clamp_urge(
                        self.urge_state.get(soul, 7.0) + 2.0 * conf)
                    self.threshold_state[soul]  = self._clamp_threshold(
                        self.threshold_state.get(soul, 6.0) - 0.2 * conf)

            elif event_type == 'STAKES':
                # Out-of-room authority — everyone with stake in scene activates
                for soul in active_souls:
                    if soul != speaker:
                        self.urge_state[soul] = self._clamp_urge(
                            self.urge_state.get(soul, 7.0) + 1.5 * conf)
                        self.threshold_state[soul] = self._clamp_threshold(
                            self.threshold_state.get(soul, 6.0) - 0.2 * conf)

            # AMBIENT: no delta applied — passive increment handles it

    def post_turn(self, turn_text, speaker, active_souls):
        """
        Called by T111 after each turn.
        Classifies event, applies deltas, updates speaker tracking.
        Returns classified events list for console logging.
        """
        events = self.classify_event(turn_text, speaker, active_souls)
        self.apply_event(events, speaker, active_souls)
        # Update speaker tracking for next turn's pronoun resolution
        self._last_speaker = speaker
        self._recent_speakers.append(speaker)
        return events

    # ══════════════════════════════════════════════════════════════════════════
    # SECTION 6: SINGLE TURN EXECUTION
    # ══════════════════════════════════════════════════════════════════════════

    def run_turn(self, actor):
        """
        Single full turn for one actor.
        Returns: (turn_text: str, scene_end: bool, new_urge: int)
        v5.5.2 — BL-E08: reads character_briefing from active_config,
        passes to vault.get_full_persona() as private knowledge block.
        """
        ui = self.r.get('ui_cmd')
        if not ui:
            raise Exception("Registry Error: 'ui_cmd' not found.")

        env = ui.env_box.value
        obj = ui.obj_box.value

        history = self.r['state'].state.get('global_history', [])
        context = self.r['slider'].slide(history, objective=obj)

        scene_role         = ""
        character_briefing = ""
        cfg = ui.active_config.get(actor, {})
        if isinstance(cfg, dict):
            role_widget     = cfg.get('role')
            briefing_widget = cfg.get('briefing')
            scene_role         = role_widget.value.strip()     if role_widget     else ""
            character_briefing = briefing_widget.value.strip() if briefing_widget else ""

        persona = self.r['vault'].get_full_persona(
            actor, environment=env, scene_role=scene_role,
            character_briefing=character_briefing)

        # BL-E07: detect passive/observer role ───────────────────────────────
        passive_mode = any(
            kw in scene_role.lower()
            for kw in ("passive", "observer", "observing", "silent observer")
        )

        response, urge, tokens, scene_end = self.r['brain'].generate(
            actor, persona, context, env, obj, passive_mode=passive_mode)

        # Passive turns cannot trigger scene end ──────────────────────────────
        if passive_mode:
            scene_end = False

        clean_response = self.r['brain'].strip_scene_end(response)
        self.r['sentinel'].audit_turn(actor, tokens)

        turn_text = f"{actor}: {clean_response}"
        self.r['state'].log_event(turn_text)

        return turn_text, scene_end, urge
# ------------------------------------------
# IPO CHECK:
# Input:   actor name (str) via run_turn()
# Process: Margin-based speaker selection → run_turn → post_turn event
#          classification → urge+threshold delta application
# Output:  (turn_text, scene_end, new_urge) from run_turn()
#
# State dicts (all managed by T111 on Start + per-turn):
#   urge_state       {actor: float 1-10}
#   threshold_state  {actor: float 3-9}
#   domain_map       {keyword: [actors]}
#   _last_speaker    str  — pronoun "you" resolution
#   _recent_speakers deque(3) — bystander proximity
# ------------------------------------------


In [44]:
# ==========================================
# T115 : GEMINI_PROV
# ==========================================
# VERSION: 2.2.0 | STATUS: Stable
# ROLE: google-genai Provider — Safe Text Extraction
# Version Remarks: Unchanged from v1.8.0.
# ------------------------------------------

from google import genai as _genai_module

class GeminiProvider:
    def __init__(self, api_key: str, model_name: str, client=None, log_fn=None):
        self.model_id = model_name.replace("models/", "")
        self.log_fn   = log_fn if log_fn else print
        if client is not None:
            self.client = client
            print(f"✅ T115: Reusing T000 Client → {self.model_id}")
        else:
            self.client = _genai_module.Client(api_key=api_key)
            print(f"✅ T115: New Client created → {self.model_id}")

    def set_log_fn(self, fn):
        self.log_fn = fn

    def _safe_extract_text(self, response) -> str:
        try:
            text = response.text
            if text: return text
        except (ValueError, AttributeError):
            pass
        try:
            parts = response.candidates[0].content.parts
            collected = [getattr(p, "text", None) for p in parts]
            text = "\n".join(t for t in collected if t)
            if text: return text
        except (IndexError, AttributeError, TypeError):
            pass
        self.log_fn("⚠️ T115: Could not extract text from response.")
        return ""

    def generate_content(self, prompt: str):
        try:
            raw = self.client.models.generate_content(
                model=self.model_id, contents=prompt)
            usage     = getattr(raw, "usage_metadata",
                                type("U",(),{"prompt_token_count":0,"candidates_token_count":0})())
            safe_text = self._safe_extract_text(raw)
            class Envelope: pass
            env = Envelope()
            env.text = safe_text; env.candidates = getattr(raw, "candidates", [])
            env.usage_metadata = usage; env._raw = raw
            return env, usage
        except Exception as e:
            self.log_fn(f"❌ T115 Call Error: {type(e).__name__}: {e}")
            class FailShield: pass
            f = FailShield()
            f.text = ""; f.candidates = []
            u = type("U",(),{"prompt_token_count":0,"candidates_token_count":0})()
            return f, u

# ------------------------------------------
# IPO CHECK:
# Input:   prompt string
# Process: generate_content() → safe extract → Envelope wrap
# Output:  (Envelope with .text str, usage_metadata)
# ------------------------------------------


In [45]:
# ==========================================
# T116 : SMOKE_TESTER
# ==========================================
# VERSION: 1.2.3 | STATUS: Stable
# ROLE: End-to-End System Integrity Validation (BFT)
# Version Remarks: v1.2.3 — BL-E09 Sprint 2.
#   _test_api() updated to unpack 4-value return from brain.generate().
#   No functional change — BFT still tests registry, state, and API handshake.
# ------------------------------------------

class SmokeTester:
    def __init__(self, registry):
        self.r = registry

    def run_bft(self):
        print("🧪 T116: INITIALIZING FULL SYSTEM BFT...")
        results = {
            "Registry Check": self._test_registry(),
            "State Write":    self._test_state(),
            "API Handshake":  self._test_api()
        }
        print("\n--- BFT REPORT ---")
        all_passed = True
        for test, passed in results.items():
            status = "✅" if passed else "❌"
            print(f"{status} {test}: {'PASSED' if passed else 'FAILED'}")
            if not passed: all_passed = False
        if not all_passed:
            print("\n🚨 CRITICAL: BFT FAILED. DO NOT PROCEED TO UI.")
        return all_passed

    def _test_registry(self):
        keys = ['state', 'brain', 'gateway', 'shield', 'sentinel', 'slider', 'vault']
        missing = [k for k in keys if k not in self.r]
        if missing:
            print(f"   Missing registry keys: {missing}")
            return False
        return True

    def _test_state(self):
        try:
            souls = self.r['state'].load_souls()
            self.r['state'].save_souls(souls)
            return True
        except Exception as e:
            print(f"   [STATE ERROR]: {e}")
            return False

    def _test_api(self):
        try:
            txt, urge, tokens, scene_end = self.r['brain'].generate(
                "System", "Test actor", "Test context", "Test env", "Ping"
            )
            return bool(txt and txt.strip())
        except Exception as e:
            print(f"   [API ERROR]: {e}")
            return False

# ------------------------------------------
# IPO CHECK:
# Input:   registry
# Process: Registry check → state round-trip → live API ping
# Output:  all_passed bool + printed BFT report
# ------------------------------------------


In [46]:
# ==========================================
# T117 : UI_CHASSIS
# ==========================================
# VERSION: 5.1.3 | STATUS: Stable
# ROLE: Master Layout & Tab Management
# Version Remarks: Unchanged from v1.8.0.
# ------------------------------------------

import ipywidgets as widgets
from IPython.display import display

class UIChassis:
    def __init__(self, sov_tab, oracle_tab, metrics_tab):
        self.sov     = sov_tab
        self.oracle  = oracle_tab
        self.metrics = metrics_tab
        self.tab     = None

    def assemble(self):
        self.tab = widgets.Tab()
        self.tab.children = [
            self.sov.get_render_box(),
            self.oracle.get_render_box(),
            self.metrics.get_render_box()
        ]
        self.tab.set_title(0, '🎮 Command Center')
        self.tab.set_title(1, '👁️ Oracle')
        self.tab.set_title(2, '📊 Metrics')

    def display(self):
        if self.tab: display(self.tab)

    def render(self):
        pass

# ------------------------------------------
# IPO CHECK:
# Input:   CommandCenter, UIOracleTab, UIMetrics instances
# Process: Wrap in Tab widget, assign titles
# Output:  Rendered tabbed UI
# ------------------------------------------


In [47]:
# ==========================================
# NOAH'S ARK v2.4.0: MASTER ASSEMBLY — Sprint 7
# ==========================================
# Chapter 2.8 — Scene Intelligence
# Sprint 7: BL-N01 T119 Scene Wizard + BL-N02 T120 Orchestrator + BL-Q05
# ------------------------------------------

# ── T118 + T000: Boot ─────────────────────────────────────────────────────────
env_loader = EnvLoader()
provider, model_name, genai_client = boot_sequence(env_loader)
api_key = env_loader.get("GEMINI_API_KEY") or env_loader.get("AGISK_Default")
initialize_env(provider)

# ── Core State + Shield + Provider ────────────────────────────────────────────
state    = StateEngine()
shield   = QuotaShield()
gateway  = GeminiProvider(api_key, model_name, client=genai_client)
vault    = IdentityVault(state)

# ── Processing Tiles ──────────────────────────────────────────────────────────
sentinel  = TokenSentinel()
archiver  = LogArchiver()
slider    = ContextSlider()
maya      = MayaMetaObserver(gateway, shield)

# ── Registry Bootstrap ────────────────────────────────────────────────────────
registry  = {}
brain     = CharacterBrain(gateway, shield, sentinel, registry)
conductor = SimConductor(registry)

registry.update({
    'state':     state,
    'shield':    shield,
    'gateway':   gateway,
    'vault':     vault,
    'sentinel':  sentinel,
    'archiver':  archiver,
    'slider':    slider,
    'brain':     brain,
    'maya':      maya,
    'conductor': conductor,
})

# ── UI Components ─────────────────────────────────────────────────────────────
ui_cmd    = CommandCenter(registry)
ui_oracle = UIOracleTab()
ui_met    = UIMetrics()

registry['ui_cmd']    = ui_cmd
registry['conductor'] = conductor

# ── Link UI to Logic ──────────────────────────────────────────────────────────
oracle_logic        = OracleLogic(registry)
registry['oracle']  = oracle_logic
ui_oracle.r         = registry
ui_met.sentinel_handle = sentinel

# ── T116 SmokeTester — BL-I05 ─────────────────────────────────────────────────
tester = SmokeTester(registry)
registry['tester'] = tester

# ── T119 Scene Wizard — BL-N01 ────────────────────────────────────────────────
wizard = SceneWizard(gateway, shield)
registry['wizard'] = wizard

# ── T120 Orchestrator — BL-N02 ────────────────────────────────────────────────
orchestrator = Orchestrator()
registry['orchestrator'] = orchestrator

# ── Final Assembly (T117) ─────────────────────────────────────────────────────
chassis = UIChassis(ui_cmd, ui_oracle, ui_met)
chassis.assemble()

# ── Global Tab Observer ───────────────────────────────────────────────────────
def system_observer(change):
    if change['new'] == 2:
        ui_met.refresh_data()
    elif change['new'] == 0:
        if hasattr(ui_cmd, '_refresh_cockpit'):
            ui_cmd._refresh_cockpit()

chassis.tab.observe(system_observer, names='selected_index')

# ── Wire Console Logging ──────────────────────────────────────────────────────
ui_cmd.wire_logging()

# 🚀 LAUNCH
chassis.display()


--- NOAH'S ARK OS v1.9.0: google-genai BOOT ---
✅ T000: google-genai Client created.
✅ T000: Discovered 28 Gemini models.
🚀 T000: Model locked → gemini-2.5-flash
✅ T100: google-genai SDK verified.
✅ T115: Reusing T000 Client → gemini-2.5-flash
